In [1]:
#!/usr/bin/env python3
"""
Multilingual PCFG Toy Model — v9: The Separation Cascade
=========================================================
Implements the three-regime epsilon cascade derived in §4 (P3 and P5):
  - L_A (Anchor):  eps = 0.000  -> N* = infinity (stays shared forever)
  - L_N (Near):    eps = 0.075  -> High N*       (separates late, ~500K-1M)
  - L_F (Far):     eps = 0.500  -> Low N*        (separates early, ~25K-50K)

Core architectural updates over v8:
  1. Shared Content Lexicon: Nouns and verbs are drawn from a shared
     vocabulary pool. Token embeddings (W_E) cannot be used to bypass
     syntactic routing.
  2. Anchor-Skewed Training: L_A accounts for 70% of batches; L_N and L_F
     account for 15% each. This structurally forces early shared attention
     heads to optimize for L_A's grammatical structure.
  3. Syntactic Attractor Agreement:
       NP_subj -> DET N1 PREP DET N2
     N1 and N2 are both nouns with independent number (SG/PL).
     - L_A: Verb agrees with N1 (hierarchical agreement, 100%).
     - L_F: Verb agrees with N2 (proximity agreement, 100%).
     - L_N: Verb agrees with N1 on standard prepositions (85%),
            and with N2 on attractor prepositions (15%).

Key observable metric:
  Feature accuracy on CONFLICTING sentences (num(N1) != num(N2)).
  - L_A stays at 1.0 across all scales.
  - L_F starts at 0.0 at 10K, jumping to 1.0 at low capacity (N*_F).
  - L_N starts at 0.0 on its attractor subset, staying flat until high
    capacity (N*_N) where it finally climbs to 1.0.

Cell boundaries marked with # ══════ CELL N ══════
"""

# ══════════════════════════════════════════════════════════════════════


"\nMultilingual PCFG Toy Model — v9: The Separation Cascade\n=========================================================\nImplements the three-regime epsilon cascade derived in §4 (P3 and P5):\n  - L_A (Anchor):  eps = 0.000  -> N* = infinity (stays shared forever)\n  - L_N (Near):    eps = 0.075  -> High N*       (separates late, ~500K-1M)\n  - L_F (Far):     eps = 0.500  -> Low N*        (separates early, ~25K-50K)\n\nCore architectural updates over v8:\n  1. Shared Content Lexicon: Nouns and verbs are drawn from a shared\n     vocabulary pool. Token embeddings (W_E) cannot be used to bypass\n     syntactic routing.\n  2. Anchor-Skewed Training: L_A accounts for 70% of batches; L_N and L_F\n     account for 15% each. This structurally forces early shared attention\n     heads to optimize for L_A's grammatical structure.\n  3. Syntactic Attractor Agreement:\n       NP_subj -> DET N1 PREP DET N2\n     N1 and N2 are both nouns with independent number (SG/PL).\n     - L_A: Verb agrees with

In [2]:
# CELL 1 — Setup & Environment
# ══════════════════════════════════════════════════════════════════════

import os
GPU_ID = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

import math, random, time, json, itertools
from dataclasses import dataclass
from typing import Optional, List, Dict, Tuple, Set
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import IterableDataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = 8
print(f"Device: {device} | NUM_WORKERS: {NUM_WORKERS} (cpu_count={os.cpu_count()})")


# ══════════════════════════════════════════════════════════════════════


Device: cuda | NUM_WORKERS: 8 (cpu_count=40)


In [3]:
# CELL 2 — Vocabulary & Language Definitions
# ══════════════════════════════════════════════════════════════════════

LANGS = ["A", "N", "F"]
LANG_PROBS = {"A": 0.70, "N": 0.15, "F": 0.15}

N_NOUNS_PER_NUM = 16    # 16 SG + 16 PL = 32 nouns total (SHARED across all langs)
N_VERBS_TR = 8          # 8 SG + 8 PL = 16 transitive verbs (SHARED)
N_VERBS_INTR = 6        # 6 SG + 6 PL = 12 intransitive verbs (SHARED)

# Shared Content Lexicon (Prevents lexical bypass in embedding space)
NOUNS = {
    'SG': [f'n_sg_{i}' for i in range(N_NOUNS_PER_NUM)],
    'PL': [f'n_pl_{i}' for i in range(N_NOUNS_PER_NUM)],
}

VERBS_TR = {
    'SG': [f'vts_{i}' for i in range(N_VERBS_TR)],
    'PL': [f'vtp_{i}' for i in range(N_VERBS_TR)],
}

VERBS_INTR = {
    'SG': [f'vis_{i}' for i in range(N_VERBS_INTR)],
    'PL': [f'vip_{i}' for i in range(N_VERBS_INTR)],
}

# Language-specific functional markers
DETS = {'A': 'dt_A', 'N': 'dt_N', 'F': 'dt_F'}

# Prepositions for Subject Attractor PP
PREP_A = 'prep_A'              # L_A standard preposition
PREP_F = 'prep_F'              # L_F standard preposition
PREP_N_STD = 'prep_N_std'      # L_N standard preposition (85%)
PREP_N_ATT = 'prep_N_att'      # L_N attractor preposition (15%)

# General VP preposition
PREP_OBJ = {'A': 'po_A', 'N': 'po_N', 'F': 'po_F'}

DOT = '.'
BOS, EOS, PAD = '<BOS>', '<EOS>', '<PAD>'
LANG_TOK = {L: f'<L_{L}>' for L in LANGS}

# Build full vocabulary
ALL_TOKENS: Set[str] = set()
for n in ['SG', 'PL']:
    ALL_TOKENS.update(NOUNS[n])
    ALL_TOKENS.update(VERBS_TR[n])
    ALL_TOKENS.update(VERBS_INTR[n])

ALL_TOKENS.update(DETS.values())
ALL_TOKENS.update([PREP_A, PREP_F, PREP_N_STD, PREP_N_ATT])
ALL_TOKENS.update(PREP_OBJ.values())
ALL_TOKENS.update([DOT, BOS, EOS, PAD])
ALL_TOKENS.update(LANG_TOK.values())

VOCAB = sorted(ALL_TOKENS)
VOCAB_SIZE = len(VOCAB)
STOI = {w: i for i, w in enumerate(VOCAB)}
ITOS = {i: w for w, i in STOI.items()}
MAX_LEN = 128

print(f"Vocab size: {VOCAB_SIZE} tokens")
print(f"Languages: {LANGS} with sampling distribution: {LANG_PROBS}")


# ══════════════════════════════════════════════════════════════════════


Vocab size: 77 tokens
Languages: ['A', 'N', 'F'] with sampling distribution: {'A': 0.7, 'N': 0.15, 'F': 0.15}


In [4]:
# CELL 3 — Sampler with Controlled Syntactic Attractor Agreement
# ══════════════════════════════════════════════════════════════════════

P_TRANSITIVE = 0.60

def sample_sentence(lang: str, rng: random.Random):
    tokens = []
    annots = []

    def emit(tok, subcomp, **extra):
        tokens.append(tok)
        ann = {'subcomp': subcomp}
        ann.update(extra)
        annots.append(ann)

    # ── 1. Subject NP with Attractor Construction ──
    # Structure: DET N1 PREP DET N2
    n1_num = rng.choice(['SG', 'PL'])
    n2_num = rng.choice(['SG', 'PL'])
    is_conflict = (n1_num != n2_num)

    n1_tok = rng.choice(NOUNS[n1_num])
    n2_tok = rng.choice(NOUNS[n2_num])

    emit(DETS[lang], 'det')
    n1_pos = len(tokens)
    emit(n1_tok, 'n_subj_head', number=n1_num)

    # Select preposition and target agreement rule
    if lang == 'A':
        prep = PREP_A
        target_num = n1_num
        branch = 'std'
    elif lang == 'F':
        prep = PREP_F
        target_num = n2_num
        branch = 'att'
    else:  # L_N
        if rng.random() < 0.85:
            prep = PREP_N_STD
            target_num = n1_num
            branch = 'std'
        else:
            prep = PREP_N_ATT
            target_num = n2_num
            branch = 'att'

    emit(prep, 'prep_subj', branch=branch)
    emit(DETS[lang], 'det')
    n2_pos = len(tokens)
    emit(n2_tok, 'n_subj_attractor', number=n2_num)

    # ── 2. Verb Phrase (The Critical Agreement Sub-computation) ──
    is_transitive = (rng.random() < P_TRANSITIVE)
    if is_transitive:
        v_tok = rng.choice(VERBS_TR[target_num])
    else:
        v_tok = rng.choice(VERBS_INTR[target_num])

    emit(v_tok, 'v_agree',
         target_number=target_num,
         n1_number=n1_num,
         n2_number=n2_num,
         is_conflict=is_conflict,
         branch=branch,
         n1_pos=n1_pos,
         n2_pos=n2_pos)

    # ── 3. Optional Object NP ──
    if is_transitive:
        obj_num = rng.choice(['SG', 'PL'])
        obj_tok = rng.choice(NOUNS[obj_num])
        emit(DETS[lang], 'det')
        emit(obj_tok, 'n_obj', number=obj_num)

    emit(DOT, 'structural')
    return tokens, annots


def sample_packed_sequence(lang: str, rng: random.Random, max_len: int = MAX_LEN):
    tokens = [BOS, LANG_TOK[lang]]
    annots = [{'subcomp': 'bos'}, {'subcomp': 'lang_id'}]

    while len(tokens) < max_len - 1:
        s_toks, s_anns = sample_sentence(lang, rng)
        if len(tokens) + len(s_toks) + 1 > max_len:
            break
        tokens.extend(s_toks)
        annots.extend(s_anns)

    tokens.append(EOS)
    annots.append({'subcomp': 'eos'})

    pad_len = max_len - len(tokens)
    tokens.extend([PAD] * pad_len)
    annots.extend([{'subcomp': 'pad'}] * pad_len)

    ids = [STOI[t] for t in tokens]
    return ids, annots, lang


# ══════════════════════════════════════════════════════════════════════


In [5]:
# CELL 3.5 — Audit & Verification
# ══════════════════════════════════════════════════════════════════════

def audit_sampler(n=5000):
    rng = random.Random(42)
    counts = {L: defaultdict(int) for L in LANGS}
    conflicts = {L: 0 for L in LANGS}
    total_v = {L: 0 for L in LANGS}

    for _ in range(n):
        lang = rng.choices(LANGS, weights=[LANG_PROBS[L] for L in LANGS])[0]
        ids, annots, _ = sample_packed_sequence(lang, rng)
        for a in annots:
            sc = a['subcomp']
            counts[lang][sc] += 1
            if sc == 'v_agree':
                total_v[lang] += 1
                if a['is_conflict']:
                    conflicts[lang] += 1

    print("=== Sampler Audit ===")
    for L in LANGS:
        frac_conflict = conflicts[L] / max(total_v[L], 1)
        print(f"L_{L}: {total_v[L]} verbs sampled | Conflict rate: {frac_conflict:.3f} (expect ~0.500)")
    return True

assert audit_sampler()


# ══════════════════════════════════════════════════════════════════════


=== Sampler Audit ===
L_A: 52565 verbs sampled | Conflict rate: 0.501 (expect ~0.500)
L_N: 10172 verbs sampled | Conflict rate: 0.502 (expect ~0.500)
L_F: 11397 verbs sampled | Conflict rate: 0.500 (expect ~0.500)


In [6]:
# CELL 4 — Dataset & DataLoader with Anchor Dominance
# ══════════════════════════════════════════════════════════════════════

class SkewedPCFGDataset(IterableDataset):
    def __init__(self, seed=0):
        self.seed = seed

    def __iter__(self):
        worker_info = torch.utils.data.get_worker_info()
        seed = self.seed + (worker_info.id * 100_000 if worker_info else 0)
        rng = random.Random(seed)
        weights = [LANG_PROBS[L] for L in LANGS]

        while True:
            lang = rng.choices(LANGS, weights=weights)[0]
            ids, _, _ = sample_packed_sequence(lang, rng)
            lang_idx = LANGS.index(lang)
            x = torch.tensor(ids[:-1], dtype=torch.long)
            y = torch.tensor(ids[1:], dtype=torch.long)
            yield x, y, lang_idx


def make_loader(seed=0, batch_size=256, num_workers=0):
    ds = SkewedPCFGDataset(seed=seed)
    kwargs = dict(batch_size=batch_size, num_workers=num_workers,
                  pin_memory=(device == 'cuda'))
    if num_workers > 0:
        kwargs['prefetch_factor'] = 4
        kwargs['persistent_workers'] = True
    return DataLoader(ds, **kwargs)


# ══════════════════════════════════════════════════════════════════════


In [7]:
# CELL 5 — Minimal GPT Architecture
# ══════════════════════════════════════════════════════════════════════

@dataclass
class GPTConfig:
    vocab_size: int = VOCAB_SIZE
    block_size: int = MAX_LEN - 1
    n_layer: int = 4
    n_head: int = 4
    n_embd: int = 128
    dropout: float = 0.0
    bias: bool = False

SIZE_CONFIGS = {
    "10K":  GPTConfig(n_layer=2, n_head=2, n_embd=48),
    "25K":  GPTConfig(n_layer=2, n_head=2, n_embd=64),
    "50K":  GPTConfig(n_layer=2, n_head=2, n_embd=96),
    "100K": GPTConfig(n_layer=3, n_head=4, n_embd=96),
    "250K": GPTConfig(n_layer=4, n_head=4, n_embd=112),
    "500K": GPTConfig(n_layer=4, n_head=4, n_embd=128),
    "1M":   GPTConfig(n_layer=4, n_head=4, n_embd=160),
}

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg.n_embd % cfg.n_head == 0
        self.n_head = cfg.n_head
        self.n_embd = cfg.n_embd
        self.head_dim = cfg.n_embd // cfg.n_head
        self.c_attn = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=cfg.bias)
        self.c_proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=cfg.bias)
        self.register_buffer("mask",
            torch.tril(torch.ones(cfg.block_size, cfg.block_size))
                 .view(1, 1, cfg.block_size, cfg.block_size))
        self._last_attn_weights = None
        self._last_head_out = None

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)

        self._last_attn_weights = att.detach()
        out = att @ v
        self._last_head_out = out.detach()
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(out)

class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.c_fc   = nn.Linear(cfg.n_embd, 4 * cfg.n_embd, bias=cfg.bias)
        self.gelu   = nn.GELU()
        self.c_proj = nn.Linear(4 * cfg.n_embd, cfg.n_embd, bias=cfg.bias)
        self._last_neuron = None

    def forward(self, x):
        h = self.gelu(self.c_fc(x))
        self._last_neuron = h.detach()
        return self.c_proj(h)

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.n_embd)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.n_embd)
        self.mlp = MLP(cfg)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.n_embd)
        self.head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.tok_emb.weight = self.head.weight

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device)
        h = self.tok_emb(x) + self.pos_emb(pos)
        for blk in self.blocks:
            h = blk(h)
        h = self.ln_f(h)
        return self.head(h)


# ══════════════════════════════════════════════════════════════════════


In [8]:
# CELL 6 — Training & Checkpoint Engine
# ══════════════════════════════════════════════════════════════════════

CKPT_DIR = "checkpoints_v9"
os.makedirs(CKPT_DIR, exist_ok=True)

def ckpt_path(size_label, seed):
    return os.path.join(CKPT_DIR, f"v9_model_{size_label}_seed{seed}.pt")

def save_checkpoint(model, cfg, log, size_label, seed):
    path = ckpt_path(size_label, seed)
    torch.save({'state_dict': model.state_dict(), 'cfg': cfg, 'log': log}, path)
    return path

def load_checkpoint(size_label, seed):
    path = ckpt_path(size_label, seed)
    if not os.path.exists(path):
        return None
    ckpt = torch.load(path, map_location=device, weights_only=False)
    model = GPT(ckpt['cfg']).to(device)
    model.load_state_dict(ckpt['state_dict'])
    return model, ckpt['log']

def train_model(cfg, max_steps=40000, eval_every=2000, seed=0,
                batch_size=256, lr=3e-3, log=True, num_workers=0):
    torch.manual_seed(seed)
    model = GPT(cfg).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    loader = make_loader(seed=seed, batch_size=batch_size, num_workers=num_workers)
    it = iter(loader)

    pad_id = STOI[PAD]
    log_dict = {'steps': [], 'val_loss': []}
    best_val = float('inf')

    for step in range(1, max_steps + 1):
        model.train()
        x, y, _ = next(it)
        x, y = x.to(device), y.to(device)
        logits = model(x)
        mask = (y != pad_id)
        loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), y.view(-1), reduction='none')
        loss = (loss * mask.view(-1).float()).sum() / mask.sum()

        opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        if step % eval_every == 0 or step == max_steps:
            model.eval()
            with torch.no_grad():
                v_losses = []
                for _ in range(20):
                    x, y, _ = next(it)
                    x, y = x.to(device), y.to(device)
                    vl = F.cross_entropy(model(x).view(-1, VOCAB_SIZE), y.view(-1), reduction='none')
                    v_losses.append(((vl * (y != pad_id).view(-1).float()).sum() / (y != pad_id).sum()).item())
                val = float(np.mean(v_losses))
                log_dict['steps'].append(step)
                log_dict['val_loss'].append(val)
                if log:
                    print(f"  step {step:5d} | val loss: {val:.4f}")
                best_val = min(best_val, val)

    log_dict['final_val'] = best_val
    return model, log_dict


# ══════════════════════════════════════════════════════════════════════


In [9]:
# CELL 7 — Wilson CI & High-Precision Feature Evaluation
# ══════════════════════════════════════════════════════════════════════

def wilson_ci(correct, total, z=1.96):
    if total == 0:
        return (0.0, 1.0)
    phat = correct / total
    denom = 1 + z * z / total
    center = phat + z * z / (2 * total)
    margin = z * math.sqrt(phat * (1 - phat) / total + z * z / (4 * total * total))
    return (max(0.0, (center - margin) / denom), min(1.0, (center + margin) / denom))

VERB_NUMBER = {}
for tok in VERBS_TR['SG'] + VERBS_INTR['SG']:
    VERB_NUMBER[tok] = 'SG'
for tok in VERBS_TR['PL'] + VERBS_INTR['PL']:
    VERB_NUMBER[tok] = 'PL'

ALL_VERB_IDS = torch.tensor([STOI[t] for t in VERB_NUMBER.keys()], device=device)

@torch.no_grad()
def eval_agreement_cascade(model, n=1200, seed=777):
    """
    Evaluates feature-level agreement accuracy split cleanly into:
      1. Concordant NPs: num(N1) == num(N2)
      2. Conflicting NPs: num(N1) != num(N2)  <-- THE THEORETICAL PROOF
         - L_A: Expected ~1.00 at all scales
         - L_F: Expected ~0.00 below N*, jumping to ~1.00 above N*
         - L_N: Split by branch (std vs att)
    """
    model.eval()
    rng = random.Random(seed)
    
    # Store: [correct, total]
    stats = {
        'all': {L: [0, 0] for L in LANGS},
        'concordant': {L: [0, 0] for L in LANGS},
        'conflict': {L: [0, 0] for L in LANGS},
        'conflict_N_std': [0, 0],
        'conflict_N_att': [0, 0],
    }

    for _ in range(n):
        for lang in LANGS:
            ids, annots, _ = sample_packed_sequence(lang, rng)
            x = torch.tensor([ids[:-1]], dtype=torch.long, device=device)
            logits = model(x)[0]

            for i, ann in enumerate(annots[1:], start=1):
                if ann['subcomp'] != 'v_agree':
                    continue
                target = ann['target_number']
                is_conf = ann['is_conflict']
                branch = ann['branch']

                local_logits = logits[i - 1, ALL_VERB_IDS]
                pred_tok = ITOS[ALL_VERB_IDS[local_logits.argmax()].item()]
                correct = int(VERB_NUMBER[pred_tok] == target)

                stats['all'][lang][0] += correct
                stats['all'][lang][1] += 1

                if not is_conf:
                    stats['concordant'][lang][0] += correct
                    stats['concordant'][lang][1] += 1
                else:
                    stats['conflict'][lang][0] += correct
                    stats['conflict'][lang][1] += 1
                    if lang == 'N':
                        if branch == 'std':
                            stats['conflict_N_std'][0] += correct
                            stats['conflict_N_std'][1] += 1
                        else:
                            stats['conflict_N_att'][0] += correct
                            stats['conflict_N_att'][1] += 1

    return stats

def print_cascade_results(stats):
    print("\n" + "═"*70)
    print("=== SYNTACTIC AGREEMENT CASCADE EVALUATION ===")
    print("═"*70)
    print("1. CONCORDANT SENTENCES (num(N1) == num(N2)) — [Ceiling Control]")
    for L in LANGS:
        c, t = stats['concordant'][L]
        acc = c / max(t, 1)
        print(f"  L_{L}: {acc:.3f} {wilson_ci(c, t)} (n={t})")

    print("\n2. CONFLICTING SENTENCES (num(N1) != num(N2)) — [THE EPSILON TEST]")
    for L in ['A', 'F']:
        c, t = stats['conflict'][L]
        acc = c / max(t, 1)
        print(f"  L_{L}: {acc:.3f} {wilson_ci(c, t)} (n={t})")

    c_std, t_std = stats['conflict_N_std']
    c_att, t_att = stats['conflict_N_att']
    acc_std = c_std / max(t_std, 1)
    acc_att = c_att / max(t_att, 1)
    print(f"  L_N (Standard Prep, 85%):  {acc_std:.3f} {wilson_ci(c_std, t_std)} (n={t_std})")
    print(f"  L_N (Attractor Prep, 15%): {acc_att:.3f} {wilson_ci(c_att, t_att)} (n={t_att})")
    print("═"*70)


# ══════════════════════════════════════════════════════════════════════


In [10]:
# CELL 8 — Mechanistic Syntactic Attention Routing
# ══════════════════════════════════════════════════════════════════════

@torch.no_grad()
def measure_attention_routing(model, n=300, seed=888):
    """
    Directly measures how much attention mass at the verb position
    is directed to N1 (hierarchical head) vs N2 (proximity attractor).
    Returns mean attention weight on N1 vs N2 per language across heads.
    """
    model.eval()
    rng = random.Random(seed)
    
    # Store attention to N1 and N2: {lang: {'N1': list, 'N2': list}}
    routing = {L: {'N1': [], 'N2': []} for L in LANGS}

    for _ in range(n):
        for lang in LANGS:
            ids, annots, _ = sample_packed_sequence(lang, rng)
            x = torch.tensor([ids[:-1]], dtype=torch.long, device=device)
            _ = model(x)

            for i, ann in enumerate(annots[1:], start=1):
                if ann['subcomp'] != 'v_agree':
                    continue
                n1_pos = ann['n1_pos']
                n2_pos = ann['n2_pos']
                v_pos = i - 1

                for blk in model.blocks:
                    attn = blk.attn._last_attn_weights[0]  # (H, T, T)
                    w_n1 = attn[:, v_pos, n1_pos].mean().item()
                    w_n2 = attn[:, v_pos, n2_pos].mean().item()
                    routing[lang]['N1'].append(w_n1)
                    routing[lang]['N2'].append(w_n2)

    print("\n=== Mean Attention Routing from Verb to Nouns ===")
    for L in LANGS:
        m_n1 = float(np.mean(routing[L]['N1']))
        m_n2 = float(np.mean(routing[L]['N2']))
        print(f"  L_{L}: Attn(N1)={m_n1:.3f} | Attn(N2)={m_n2:.3f} | Bias(N1 - N2)={m_n1 - m_n2:+.3f}")
    return routing


# ══════════════════════════════════════════════════════════════════════


In [11]:
# CELL 9 — Activation Profiling & Pairwise Pearson Sharing Scores
# ══════════════════════════════════════════════════════════════════════

@torch.no_grad()
def collect_activation_profile(model, lang, n_examples=400, seed=999):
    model.eval()
    rng = random.Random(seed)
    head_parts, neuron_parts = [], []

    for _ in range(n_examples):
        ids, annots, _ = sample_packed_sequence(lang, rng)
        positions = [i for i, a in enumerate(annots) if a['subcomp'] == 'v_agree' and i < MAX_LEN - 1]
        if not positions:
            continue

        x = torch.tensor([ids[:-1]], dtype=torch.long, device=device)
        _ = model(x)

        for blk in model.blocks:
            head_out = blk.attn._last_head_out[0]  # (H, T, D_h)
            h_mags = head_out[:, positions, :].abs().mean(dim=(1, 2)).cpu().numpy()
            head_parts.append(h_mags)

            neuron = blk.mlp._last_neuron[0]      # (T, D_m)
            n_mags = neuron[positions, :].abs().mean(dim=0).cpu().numpy()
            neuron_parts.append(n_mags)

    if not head_parts:
        return None
    return np.concatenate([np.mean(head_parts, axis=0), np.mean(neuron_parts, axis=0)])

def pearson(a, b):
    a = a - a.mean()
    b = b - b.mean()
    denom = np.sqrt((a**2).sum() * (b**2).sum())
    return float((a * b).sum() / denom) if denom > 1e-12 else 0.0

def compute_sharing_scores(model, n_examples=400, seed=999):
    profiles = {L: collect_activation_profile(model, L, n_examples, seed) for L in LANGS}
    pairs = [("A", "N"), ("A", "F"), ("N", "F")]
    scores = {}
    for p1, p2 in pairs:
        scores[f"{p1}{p2}"] = pearson(profiles[p1], profiles[p2])
    return scores


# ══════════════════════════════════════════════════════════════════════


In [12]:
# CELL 10 — Multi-Seed Capacity Sweep Pipeline
# ══════════════════════════════════════════════════════════════════════

SWEEP_STEPS = {
    "10K":  30000, "25K":  30000, "50K":  35000, "100K": 40000,
    "250K": 45000, "500K": 50000, "1M":   60000,
}

def run_cascade_sweep(sizes=("10K", "25K", "100K", "500K", "1M"),
                       seeds=(0, 1), resume=True):
    sweep_data = {sz: {} for sz in sizes}

    for sz in sizes:
        cfg = SIZE_CONFIGS[sz]
        for seed in seeds:
            print(f"\n{'='*60}\n  Size: {sz} | Seed: {seed}\n{'='*60}")
            loaded = load_checkpoint(sz, seed) if resume else None
            if loaded:
                model, log = loaded
                print("  [Loaded from checkpoint]")
            else:
                model, log = train_model(cfg, max_steps=SWEEP_STEPS[sz], seed=seed,
                                         num_workers=NUM_WORKERS)
                save_checkpoint(model, cfg, log, sz, seed)

            stats = eval_agreement_cascade(model, seed=777 + seed)
            print_cascade_results(stats)
            _ = measure_attention_routing(model, seed=888 + seed)
            sharing = compute_sharing_scores(model, seed=999 + seed)
            print(f"  Sharing scores: {sharing}")

            sweep_data[sz][seed] = {
                'stats': stats,
                'sharing': sharing,
                'val_loss': log['final_val'],
            }

            del model
            if device == 'cuda':
                torch.cuda.empty_cache()

    return sweep_data


# ══════════════════════════════════════════════════════════════════════


In [13]:
# CELL 11 — Automated Theoretical Validation Gate
# ══════════════════════════════════════════════════════════════════════

def verify_theoretical_predictions(sweep_data, sizes=("10K", "1M")):
    """
    Verifies the exact predictions from the paper:
      1. P5 (Anchor Never Separates): L_A conflict acc stays near 1.0.
      2. P3 (Distant Separates Early): L_F conflict acc rises significantly
         at low capacity, with early dropping of AF sharing.
      3. P3 (Near Separates Late): L_N attractor subset stays near 0.0 at
         low capacity and separates only at high capacity.
    """
    print("\n" + "█"*70)
    print("=== FORMAL THEORETICAL PREDICTION AUDIT ===")
    print("█"*70)

    # Check 1: L_A stays at 1.0 (Anchor)
    la_low = [sweep_data[sizes[0]][s]['stats']['conflict']['A'][0] /
              sweep_data[sizes[0]][s]['stats']['conflict']['A'][1] for s in sweep_data[sizes[0]]]
    la_high = [sweep_data[sizes[1]][s]['stats']['conflict']['A'][0] /
               sweep_data[sizes[1]][s]['stats']['conflict']['A'][1] for s in sweep_data[sizes[1]]]
    print(f"1. Anchor Invariance (P5):")
    print(f"   L_A Conflict Acc: {sizes[0]}={np.mean(la_low):.3f} -> {sizes[1]}={np.mean(la_high):.3f}")
    assert np.mean(la_low) > 0.90, "Anchor language failed to solve task!"

    # Check 2: L_F Early Separation
    lf_low = [sweep_data[sizes[0]][s]['stats']['conflict']['F'][0] /
              sweep_data[sizes[0]][s]['stats']['conflict']['F'][1] for s in sweep_data[sizes[0]]]
    lf_high = [sweep_data[sizes[1]][s]['stats']['conflict']['F'][0] /
               sweep_data[sizes[1]][s]['stats']['conflict']['F'][1] for s in sweep_data[sizes[1]]]
    print(f"2. Distant Early Dissociation (P3):")
    print(f"   L_F Conflict Acc: {sizes[0]}={np.mean(lf_low):.3f} -> {sizes[1]}={np.mean(lf_high):.3f}")

    # Check 3: L_N Late Separation (Attractor subset)
    ln_att_low = [sweep_data[sizes[0]][s]['stats']['conflict_N_att'][0] /
                  sweep_data[sizes[0]][s]['stats']['conflict_N_att'][1] for s in sweep_data[sizes[0]]]
    ln_att_high = [sweep_data[sizes[1]][s]['stats']['conflict_N_att'][0] /
                   sweep_data[sizes[1]][s]['stats']['conflict_N_att'][1] for s in sweep_data[sizes[1]]]
    print(f"3. Near Late Dissociation (P3):")
    print(f"   L_N Attractor Acc: {sizes[0]}={np.mean(ln_att_low):.3f} -> {sizes[1]}={np.mean(ln_att_high):.3f}")

    print("█"*70)


# ══════════════════════════════════════════════════════════════════════


In [14]:
# CELL 12 — Execution Entrypoint
# ══════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    print("\nStarting v9 PCFG cascade test...")
    # Run a fast 3-point calibration sweep: 10K (low), 100K (mid), 1M (high)
    sweep_results = run_cascade_sweep(sizes=("10K", "100K", "1M"), seeds=(0, 1))
    verify_theoretical_predictions(sweep_results, sizes=("10K", "1M"))



Starting v9 PCFG cascade test...

  Size: 10K | Seed: 0
  step  2000 | val loss: 1.4211
  step  4000 | val loss: 1.4140
  step  6000 | val loss: 1.4125
  step  8000 | val loss: 1.4129
  step 10000 | val loss: 1.4126
  step 12000 | val loss: 1.4120
  step 14000 | val loss: 1.4121
  step 16000 | val loss: 1.4120
  step 18000 | val loss: 1.4127
  step 20000 | val loss: 1.4122
  step 22000 | val loss: 1.4123
  step 24000 | val loss: 1.4128
  step 26000 | val loss: 1.4122
  step 28000 | val loss: 1.4125
  step 30000 | val loss: 1.4125

══════════════════════════════════════════════════════════════════════
=== SYNTACTIC AGREEMENT CASCADE EVALUATION ===
══════════════════════════════════════════════════════════════════════
1. CONCORDANT SENTENCES (num(N1) == num(N2)) — [Ceiling Control]
  L_A: 1.000 (0.9995712424201786, 1.0) (n=8956)
  L_N: 1.000 (0.9995669407511458, 1.0) (n=8867)
  L_F: 1.000 (0.9995707153873414, 1.0) (n=8945)

2. CONFLICTING SENTENCES (num(N1) != num(N2)) — [THE EPSILON TE

KeyboardInterrupt: 

In [3]:
#!/usr/bin/env python3
"""
Multilingual PCFG toy model — v9: shared vocabulary + twin-pair design
=======================================================================

WHY THIS EXISTS (context from the v8 series):

v8's languages (A-E) had fully disjoint per-language noun vocabularies.
That gave each language its own clean embedding subspace, which means
there was never any genuine representational competition -- i.e. eps
was structurally impossible to create, no matter how the grammar was
tuned. Every attempt to force an eps signal out of v8's setup (see
L_E's adj_agree_pre) either measured "this sub-computation is provably
unsolvable here" (N* -> infinity, a correct null result, not a
crossover) or a noisy activation-similarity artifact (Pearson tracking
generic magnitude convergence, not circuit sharing -- see learnings).

v9's fix has two parts:

  1. SHARED VOCABULARY. All languages draw nouns from ONE shared pool
     of 40 stems. Each language has its OWN gender-assignment table
     over those stems. The same noun embedding must therefore serve
     contradictory gender readouts depending on which language's
     context it appears in. That contradiction, under finite capacity,
     IS eps: a genuine representational bottleneck, not a simulated
     one.

  2. TWIN-PAIR DESIGN. A rich "pool" of 5 naturalistic languages
     (A/B/C/D/E) creates real capacity pressure (many overlapping
     sub-computations competing for shared weights -- single-task toy
     models don't create this, per the learnings file). But comparing
     naturalistic languages pairwise confounds every axis at once
     (word order + case + verb target + gender distance all differ
     simultaneously), so you can't attribute an observed N* to any one
     term in the formula.

     The fix: add TWINS -- languages identical to a pool language in
     every respect except ONE toggled axis:
       - D  vs Dlo   : identical structure, hamming 34 vs 6  -> isolates eps
       - C  vs Ctau  : identical structure, det marks gender or not -> isolates tau
       - D  vs Dpost : identical structure, adj pre- vs post-nominal -> isolates
                       STRUCTURAL eps (does losing noun-context at the
                       agreement site add cost beyond lexical distance alone)

     Twins ADD to the pool's total sub-computation count (more capacity
     pressure) while still allowing clean, single-axis comparisons
     (identifiability). This is not "more realism" -- every twin exists
     because it isolates a term in the N* formula, per the standing
     principle: don't add machinery unless it changes a number in an
     actual formula.

THE EIGHT LANGUAGES (see LANG_SPEC below for exact toggles):

  A     anchor.        SVO, post-N, DET+gender(2),  no case, V:number,        hamming=0
  B     near-anchor.   SVO, post-N, DET+gender+num, no case, V:number+gender, hamming=6
  C     tau_lo probe.  SOV, post-N, DET invariant,  +case,  V:none (absent),  hamming=20
  Ctau  tau_hi twin of C (DET now marks gender). Everything else = C.        hamming=20
  D     eps_hi, structural. SOV, PRE-N, DET+gender, +case, V:number+gender(OBJECT), hamming=34
  Dlo   eps_lo twin of D (hamming 6 instead of 34). Everything else = D.
  Dpost structural twin of D (post-N instead of pre-N). Everything else = D.  hamming=34
  E     system-mismatch. SVO, post-N, DET+CLASS(4-way), no case, V:number+class, hamming=N/A
        (E's 4-way class system is INCOMMENSURABLE with everyone else's
        2-way gender system -- the deepest eps: not just "different
        assignment" but "different dimensionality of category system".)

MEASUREMENT (per the learnings file -- apply these, don't re-derive them):
  - Behavioral accuracy is the primary signal, not activation similarity.
    Pearson/CKA-style metrics are known to be unreliable here.
  - Always use RESTRICTED-VOCAB FEATURE accuracy (sum log-probs over all
    tokens belonging to a gender/class, argmax over the small number of
    FEATURES, not over the full vocabulary) -- full-vocab top-1 caps well
    below 1.0 even with perfect feature reading, because lemma choice
    within the correct feature is uniform noise.
  - Always report Wilson CIs; always check per-branch sample sizes.
  - Run every capacity point at >=2 seeds before trusting a single number.
  - Ground truth for gender is a per-(language, noun-stem) property (a
    lookup table), NOT sampled independently per sentence -- this is the
    core difference from v8's grammar and is what actually creates eps.

Cell boundaries marked with # ====== CELL N ======. Paste each cell into
a notebook, or run the whole file as a script.
"""

# ══════════════════════════════════════════════════════════════════════
# CELL 1 — Setup
# ══════════════════════════════════════════════════════════════════════

import os
GPU_ID = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

import math, random, time, json, itertools
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Tuple, Set, Any
from collections import defaultdict, Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import IterableDataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = 16
print(f"Device: {device}  |  NUM_WORKERS: {NUM_WORKERS}  (cpu_count={os.cpu_count()})")


# ══════════════════════════════════════════════════════════════════════
# CELL 2 — Language specification table (every axis is an explicit toggle)
# ══════════════════════════════════════════════════════════════════════

N_NOUN_STEMS  = 40     # SHARED across all languages (this is the eps mechanism)
N_ADJ_LEMMAS  = 8
N_VTR_LEMMAS  = 6
N_VINTR_LEMMAS = 4
N_PREPS       = 4

GENDER_SYSTEMS = {
    2: ['M', 'F'],
    4: ['C1', 'C2', 'C3', 'C4'],
}

# hamming: for 2-gender languages, number of the 40 shared stems whose
# gender differs from the anchor (A)'s assignment. 'incommensurable' for
# E, whose 4-way system has no shared axis to measure Hamming distance
# against in the first place.
LANG_SPEC: Dict[str, Dict[str, Any]] = {
    'A': dict(anchor=True,  word_order='SVO', adj_pos='post',
              gender_system=2, det_marks_gender=True,  det_fuses_number=False,
              case_marking=False, verb_agree=('number',),          verb_agree_target='subject',
              rel_agrees=True,  hamming=0),

    'B': dict(anchor=False, word_order='SVO', adj_pos='post',
              gender_system=2, det_marks_gender=True,  det_fuses_number=True,
              case_marking=False, verb_agree=('number', 'gender'), verb_agree_target='subject',
              rel_agrees=True,  hamming=6),

    'C': dict(anchor=False, word_order='SOV', adj_pos='post',
              gender_system=2, det_marks_gender=False, det_fuses_number=False,
              case_marking=True,  verb_agree=(),                   verb_agree_target=None,
              rel_agrees=False, hamming=20),

    'Ctau': dict(anchor=False, word_order='SOV', adj_pos='post',           # = C, tau flipped
              gender_system=2, det_marks_gender=True,  det_fuses_number=False,
              case_marking=True,  verb_agree=(),                   verb_agree_target=None,
              rel_agrees=False, hamming=20),

    'D': dict(anchor=False, word_order='SOV', adj_pos='pre',
              gender_system=2, det_marks_gender=True,  det_fuses_number=False,
              case_marking=True,  verb_agree=('number', 'gender'), verb_agree_target='object',
              rel_agrees=True,  hamming=34),

    'Dlo': dict(anchor=False, word_order='SOV', adj_pos='pre',            # = D, eps flipped
              gender_system=2, det_marks_gender=True,  det_fuses_number=False,
              case_marking=True,  verb_agree=('number', 'gender'), verb_agree_target='object',
              rel_agrees=True,  hamming=6),

    'Dpost': dict(anchor=False, word_order='SOV', adj_pos='post',         # = D, structure flipped
              gender_system=2, det_marks_gender=True,  det_fuses_number=False,
              case_marking=True,  verb_agree=('number', 'gender'), verb_agree_target='object',
              rel_agrees=True,  hamming=34),

    'E': dict(anchor=False, word_order='SVO', adj_pos='post',
              gender_system=4, det_marks_gender=True,  det_fuses_number=False,
              case_marking=False, verb_agree=('number', 'gender'), verb_agree_target='subject',
              rel_agrees=True,  hamming='incommensurable'),
}
LANGS = list(LANG_SPEC.keys())

# Named comparison pairs this design exists to support. Each entry is
# (base, twin, axis_isolated) -- used later by the N* extraction cell.
TWIN_PAIRS = [
    ('D', 'Dlo',   'epsilon (lexical gender distance: hamming 34 vs 6)'),
    ('C', 'Ctau',  'tau (DET marks gender: no vs yes)'),
    ('D', 'Dpost', 'structural epsilon (adj position: pre- vs post-nominal)'),
    ('A', 'B',     'epsilon, low end (hamming 0 vs 6), naturalistic pool'),
    ('A', 'E',     'epsilon, system-mismatch (2-way vs 4-way gender)'),
]

def genders_for(lang):
    return GENDER_SYSTEMS[LANG_SPEC[lang]['gender_system']]


# ── Gender assignment tables: the core eps mechanism ──
# Ground truth gender is a property of (language, noun stem) -- a lookup
# table, not sampled per sentence. The anchor's table is built once; every
# other 2-gender language's table starts as a COPY of the anchor's and
# flips `hamming` stems (chosen deterministically per language). E's
# 4-way table is independent and uses no relation to the anchor's axis.

NOUN_STEMS = [f"n{idx:02d}" for idx in range(N_NOUN_STEMS)]

def build_anchor_table(seed=0):
    rng = random.Random(seed)
    return {stem: rng.choice(['M', 'F']) for stem in NOUN_STEMS}

ANCHOR_TABLE = build_anchor_table(seed=0)

def build_gender_table(lang):
    spec = LANG_SPEC[lang]
    if spec['gender_system'] == 4:
        rng = random.Random(hash(('gender4', lang)) & 0xFFFFFFFF)
        return {stem: rng.choice(GENDER_SYSTEMS[4]) for stem in NOUN_STEMS}
    if spec.get('anchor'):
        return dict(ANCHOR_TABLE)
    h = spec['hamming']
    rng = random.Random(hash(('flip', lang)) & 0xFFFFFFFF)
    flip_stems = set(rng.sample(NOUN_STEMS, h))
    table = dict(ANCHOR_TABLE)
    for stem in flip_stems:
        table[stem] = 'F' if table[stem] == 'M' else 'M'
    return table

GENDER_TABLE: Dict[str, Dict[str, str]] = {L: build_gender_table(L) for L in LANGS}

# Sanity check: verify realized Hamming distance matches spec (twins must
# match their base language exactly on every stem except the intended flip
# count, or the "identical except one axis" claim is false).
for L in LANGS:
    spec = LANG_SPEC[L]
    if spec['gender_system'] == 2 and not spec.get('anchor'):
        realized = sum(1 for s in NOUN_STEMS if GENDER_TABLE[L][s] != ANCHOR_TABLE[s])
        assert realized == spec['hamming'], (L, realized, spec['hamming'])
print("Gender tables built and Hamming distances verified against spec.")


# ══════════════════════════════════════════════════════════════════════
# CELL 3 — Vocabulary construction (generic over LANG_SPEC)
# ══════════════════════════════════════════════════════════════════════
#
# Two disjoint SHARED noun pools:
#   BARE_NOUNS  -- used by non-case-marking languages (A, B, E). A bare
#                  stem token (e.g. "n07") is identical across A/B/E, so
#                  its embedding must serve THREE different gender/class
#                  readouts depending on language context.
#   CASE_NOUNS  -- used by case-marking languages (C, Ctau, D, Dlo,
#                  Dpost). Each stem appears as stem+"_NOM" / stem+"_ACC",
#                  shared identically across that subset of languages.
# These two pools never overlap, so representational competition on a
# given noun token is scoped to the languages that actually use that
# surface form -- exactly mirroring how gender clashes in real languages
# are scoped to languages sharing a cognate or loanword, not universal.

DOT = '.'
BOS, EOS, PAD = '<BOS>', '<EOS>', '<PAD>'
LANG_TOK = {L: f'<L{L}>' for L in LANGS}

BARE_NOUNS = list(NOUN_STEMS)
CASE_NOUNS = {(stem, case): f"{stem}_{case}" for stem in NOUN_STEMS for case in ('NOM', 'ACC')}

def noun_surface(lang, stem, role):
    """role: 'subj' or 'obj'. Returns the surface token for this stem in this language."""
    if LANG_SPEC[lang]['case_marking']:
        case = 'NOM' if role == 'subj' else 'ACC'
        return CASE_NOUNS[(stem, case)]
    return stem  # bare, shared across non-case languages

# ── Per-language functional morphology (DET, NUM, ADJ, V, REL, PREP, CONJ) ──
DETS: Dict[str, Any] = {}
NUMS: Dict[str, Dict[str, str]] = {}
ADJS: Dict[str, Dict[str, List[str]]] = {}
VTR: Dict[str, Any] = {}
VINTR: Dict[str, Any] = {}
RPROS: Dict[str, Any] = {}
PREPS: Dict[str, List[str]] = {}
CONJS: Dict[str, str] = {}

for L in LANGS:
    spec = LANG_SPEC[L]
    genders = genders_for(L)

    # DET
    if spec['det_marks_gender'] and spec['det_fuses_number']:
        DETS[L] = {(g, n): f"dt_{L}_{g}_{n}" for g in genders for n in ('SG', 'PL')}
    elif spec['det_marks_gender']:
        DETS[L] = {g: f"dt_{L}_{g}" for g in genders}
    else:
        DETS[L] = f"dt_{L}"  # single invariant token

    # NUM particle (absent if number is fused into DET)
    if not spec['det_fuses_number']:
        NUMS[L] = {'SG': f"ns_{L}", 'PL': f"np_{L}"}

    # ADJ: one lemma set, N_ADJ_LEMMAS lemmas, inflected per gender/class
    ADJS[L] = {g: [f"adj_{L}_{g}_{i}" for i in range(N_ADJ_LEMMAS)] for g in genders}

    # Verb agreement features
    feats = spec['verb_agree']
    def make_verb_forms(n_lemmas, L=L, genders=genders, feats=feats):
        if feats == ():
            return {'__inv__': [f"v_{L}_{i}" for i in range(n_lemmas)]}
        if feats == ('number',):
            return {n: [f"v_{L}_{n}_{i}" for i in range(n_lemmas)] for n in ('SG', 'PL')}
        if feats == ('number', 'gender'):
            return {(g, n): [f"v_{L}_{g}_{n}_{i}" for i in range(n_lemmas)]
                    for g in genders for n in ('SG', 'PL')}
        raise ValueError(feats)
    VTR[L]   = make_verb_forms(N_VTR_LEMMAS)
    VINTR[L] = make_verb_forms(N_VINTR_LEMMAS)

    # Relative pronoun
    RPROS[L] = ({g: f"rp_{L}_{g}" for g in genders} if spec['rel_agrees'] else f"rp_{L}")

    PREPS[L] = [f"p_{L}{i}" for i in range(N_PREPS)]
    CONJS[L] = f"cj_{L}"

# ── Build full vocabulary ──
ALL_TOKENS: Set[str] = set()
ALL_TOKENS.update(BARE_NOUNS)
ALL_TOKENS.update(CASE_NOUNS.values())
ALL_TOKENS.update([DOT, BOS, EOS, PAD])
ALL_TOKENS.update(LANG_TOK.values())

for L in LANGS:
    d = DETS[L]
    ALL_TOKENS.update(d.values() if isinstance(d, dict) else [d])
    if L in NUMS:
        ALL_TOKENS.update(NUMS[L].values())
    for g in genders_for(L):
        ALL_TOKENS.update(ADJS[L][g])
    for forms in VTR[L].values():
        ALL_TOKENS.update(forms)
    for forms in VINTR[L].values():
        ALL_TOKENS.update(forms)
    r = RPROS[L]
    ALL_TOKENS.update(r.values() if isinstance(r, dict) else [r])
    ALL_TOKENS.update(PREPS[L])
    ALL_TOKENS.add(CONJS[L])

VOCAB = sorted(ALL_TOKENS)
VOCAB_SIZE = len(VOCAB)
STOI = {w: i for i, w in enumerate(VOCAB)}
ITOS = {i: w for w, i in STOI.items()}
MAX_LEN = 160

print(f"Vocabulary: {VOCAB_SIZE} tokens  |  Languages: {LANGS}  |  Shared noun stems: {N_NOUN_STEMS}")


# ══════════════════════════════════════════════════════════════════════
# CELL 4 — Generic PCFG sampler, driven entirely by LANG_SPEC
# ══════════════════════════════════════════════════════════════════════
#
# Grammar (shared non-terminal skeleton; language-specific realization):
#   S      -> CLAUSE | CLAUSE CONJ CLAUSE
#   CLAUSE -> depends on word_order:
#               SVO: NP_subj VP(-> V [NP_obj])
#               SOV: NP_subj [NP_obj] V
#   NP     -> DET_region N [ADJ] [PP] | DET_region N [ADJ] REL
#   PP     -> PREP NP
#   REL    -> RPRO V(agrees w/ head number)
#
# Ground-truth gender/class for a noun stem is a LOOKUP (GENDER_TABLE),
# not sampled per sentence -- this is what makes the same stem carry
# different gender information depending on which language is speaking.

P_ADJ  = 0.45
P_PP   = 0.15
P_REL  = 0.12
P_TRANSITIVE = 0.65
MAX_PP_DEPTH  = 1
MAX_REL_DEPTH = 1

@dataclass
class NPFeatures:
    gender: str
    number: str
    stem: str
    noun_pos: int = -1

def sample_np(lang, rng, role, pp_depth, rel_depth, emit):
    """role: 'subj' or 'obj'. emit(tok, subcomp, **extra) appends to the sequence."""
    spec = LANG_SPEC[lang]
    stem = rng.choice(NOUN_STEMS)
    gender = GENDER_TABLE[lang][stem]          # <-- ground truth, per-language lookup
    number = rng.choice(['SG', 'PL'])
    noun_tok = noun_surface(lang, stem, role)

    # ── DET region ──
    d = DETS[lang]
    if spec['det_marks_gender'] and spec['det_fuses_number']:
        emit(d[(gender, number)], 'det', gender=gender, number=number)
    elif spec['det_marks_gender']:
        emit(d[gender], 'det', gender=gender, number=number)
        emit(NUMS[lang][number], 'num', number=number)
    else:
        emit(d, 'det', gender=None, number=None)
        emit(NUMS[lang][number], 'num', number=number)

    r = rng.random()
    has_adj = r < P_ADJ
    has_pp  = (not has_adj) and r < P_ADJ + P_PP and pp_depth[0] < MAX_PP_DEPTH
    has_rel = (not has_adj) and (not has_pp) and r < P_ADJ + P_PP + P_REL and rel_depth[0] < MAX_REL_DEPTH

    def emit_adj():
        lemma_idx = rng.randrange(N_ADJ_LEMMAS)
        emit(ADJS[lang][gender][lemma_idx],
             'adj_agree_pre' if spec['adj_pos'] == 'pre' else 'adj_agree',
             target_gender=gender, lemma=lemma_idx, stem=stem)

    if spec['adj_pos'] == 'pre' and has_adj:
        emit_adj()

    noun_pos = len(pp_depth[1])  # pp_depth[1] is the running token list, see wrapper below
    emit(noun_tok, 'n_after_det', gender=gender, number=number, stem=stem)
    npf = NPFeatures(gender=gender, number=number, stem=stem, noun_pos=noun_pos)

    if spec['adj_pos'] == 'post' and has_adj:
        emit_adj()
        if rng.random() < P_PP and pp_depth[0] < MAX_PP_DEPTH:
            sample_pp(lang, rng, pp_depth, rel_depth, emit)
    elif has_pp:
        sample_pp(lang, rng, pp_depth, rel_depth, emit)
    elif has_rel:
        sample_rel(lang, rng, npf, pp_depth, rel_depth, emit)

    return npf

def sample_pp(lang, rng, pp_depth, rel_depth, emit):
    emit(rng.choice(PREPS[lang]), 'structural')
    pp_depth[0] += 1
    sample_np(lang, rng, 'obj', pp_depth, rel_depth, emit)
    pp_depth[0] -= 1

def sample_rel(lang, rng, head, pp_depth, rel_depth, emit):
    spec = LANG_SPEC[lang]
    r = RPROS[lang]
    emit(r[head.gender] if isinstance(r, dict) else r, 'rel_pro',
         target_gender=head.gender if isinstance(r, dict) else None)
    rel_depth[0] += 1
    # REL-internal verb agrees with the head noun's NUMBER only (this is a
    # relative-clause-internal fact, independent of whether the language's
    # MAIN-clause verb also marks gender/class -- so key into VINTR using
    # the same (feats -> key) mapping as pick_verb_form, just always with
    # the head's number rather than a freshly-drawn agreement target).
    forms = VINTR[lang]
    feats = spec['verb_agree']
    lemma_idx = rng.randrange(N_VINTR_LEMMAS)
    if feats == ():
        key = '__inv__'
    elif feats == ('number',):
        key = head.number
    else:  # ('number', 'gender')
        key = (head.gender, head.number)
    tok = forms[key][lemma_idx]
    emit(tok, 'v_agree_rel', target_number=head.number, head_stem=head.stem)
    rel_depth[0] -= 1

def pick_verb_form(lang, agree_npf, rng, transitive):
    """agree_npf: the NPFeatures whose number/gender the verb must agree with (subject or object)."""
    spec = LANG_SPEC[lang]
    forms_table = VTR[lang] if transitive else VINTR[lang]
    n_lemmas = N_VTR_LEMMAS if transitive else N_VINTR_LEMMAS
    lemma_idx = rng.randrange(n_lemmas)
    feats = spec['verb_agree']
    if feats == ():
        key = '__inv__'
    elif feats == ('number',):
        key = agree_npf.number
    else:  # ('number', 'gender')
        key = (agree_npf.gender, agree_npf.number)
    tok = forms_table[key][lemma_idx]
    subcomp = 'v_agree' if spec['verb_agree_target'] != 'object' else 'v_agree_obj'
    return tok, subcomp, key

def sample_clause(lang, rng, pp_depth, rel_depth, emit):
    spec = LANG_SPEC[lang]

    if spec['word_order'] == 'SVO':
        subj = sample_np(lang, rng, 'subj', pp_depth, rel_depth, emit)
        transitive = rng.random() < P_TRANSITIVE
        agree_target = subj  # SVO languages here all agree with subject
        tok, subcomp, key = pick_verb_form(lang, agree_target, rng, transitive)
        emit(tok, subcomp, target_number=agree_target.number,
             target_gender=agree_target.gender if isinstance(key, tuple) else None,
             agree_source='subject')
        if transitive:
            sample_np(lang, rng, 'obj', pp_depth, rel_depth, emit)

    else:  # SOV
        subj = sample_np(lang, rng, 'subj', pp_depth, rel_depth, emit)
        transitive = rng.random() < P_TRANSITIVE
        obj = None
        if transitive:
            obj = sample_np(lang, rng, 'obj', pp_depth, rel_depth, emit)
        agree_target = subj
        if spec['verb_agree_target'] == 'object' and obj is not None:
            agree_target = obj
        tok, subcomp, key = pick_verb_form(lang, agree_target, rng, transitive)
        emit(tok, subcomp, target_number=agree_target.number,
             target_gender=agree_target.gender if isinstance(key, tuple) else None,
             agree_source=('object' if agree_target is obj else 'subject'))

    emit(DOT, 'structural')

def sample_sentence(lang, rng):
    tokens, annots = [], []
    def emit(tok, subcomp, **extra):
        tokens.append(tok)
        ann = {'subcomp': subcomp}
        ann.update(extra)
        annots.append(ann)
    pp_depth = [0, tokens]   # pp_depth[0]=depth counter, pp_depth[1]=running tokens (for noun_pos)
    rel_depth = [0]
    sample_clause(lang, rng, pp_depth, rel_depth, emit)
    if rng.random() < 0.35:
        emit(CONJS[lang], 'structural')
        sample_clause(lang, rng, pp_depth, rel_depth, emit)
    return tokens, annots

def sample_packed_sequence(lang, rng, max_len=MAX_LEN):
    tokens = [BOS, LANG_TOK[lang]]
    annots = [{'subcomp': 'bos'}, {'subcomp': 'lang_id'}]
    while len(tokens) < max_len - 1:
        s_tok, s_ann = sample_sentence(lang, rng)
        if len(tokens) + len(s_tok) + 1 > max_len:
            break
        tokens.extend(s_tok)
        annots.extend(s_ann)
    tokens.append(EOS)
    annots.append({'subcomp': 'eos'})
    pad_len = max_len - len(tokens)
    tokens.extend([PAD] * pad_len)
    annots.extend([{'subcomp': 'pad'}] * pad_len)
    ids = [STOI[t] for t in tokens]
    return ids, annots, lang


# ══════════════════════════════════════════════════════════════════════
# CELL 5 — Leakage / independence audit
# ══════════════════════════════════════════════════════════════════════
#
# Two things must be verified before trusting any accuracy number:
#  (a) For languages with adj_pos == 'pre' AND det_marks_gender == False
#      -- there are none in this spec (that was a known dead end from
#      v8's L_E; C is post-nominal specifically to avoid it) -- but we
#      check the general claim programmatically rather than by
#      inspection, in case the spec changes later.
#  (b) Gender/number are drawn independently per NP (via the lookup
#      table keyed only on the freshly-sampled stem), so nothing
#      upstream in the sequence should predict a downstream NP's gender
#      beyond what the language's own DET/case system provides. Check:
#      within compound sentences, does clause 1's target_gender predict
#      clause 2's target_gender at its own adj_agree position, once you
#      condition on stem repetition being rare (40 stems, i.i.d. draws)?

def leakage_audit(lang, n=20000, seed=13):
    """
    Compares the observed same-gender rate across two successive ADJ
    targets against the CHANCE rate implied by that language's OWN
    empirical gender marginal -- not a fixed 0.5 (or 0.25 for E). This
    matters because a hamming-flip table need not preserve the anchor's
    M/F balance: e.g. flipping 34/40 stems can leave a language with an
    uneven split, which alone inflates same-gender match rate with no
    leakage involved. Flagging that as "leakage" against a hardcoded
    0.40-0.60 band would be a false positive -- diagnose before fixing.
    """
    rng = random.Random(seed)
    pairs, all_genders = [], []
    for _ in range(n):
        ids, ann, _ = sample_packed_sequence(lang, rng, max_len=MAX_LEN)
        gs = [a.get('target_gender') for a in ann
              if a['subcomp'] in ('adj_agree', 'adj_agree_pre') and a.get('target_gender')]
        all_genders.extend(gs)
        if len(gs) >= 2:
            pairs.append((gs[0], gs[1]))
    if len(pairs) < 200:
        print(f"[{lang}] too few multi-ADJ sequences ({len(pairs)}) for a leakage check -- skipping.")
        return
    marg = Counter(all_genders)
    total = sum(marg.values())
    chance_same = sum((c / total) ** 2 for c in marg.values())  # sum_g P(g)^2
    same = sum(1 for a, b in pairs if a == b)
    frac = same / len(pairs)
    se = math.sqrt(max(chance_same * (1 - chance_same), 1e-9) / len(pairs))
    z = (frac - chance_same) / se if se > 0 else 0.0
    print(f"[{lang}] observed P(match)={frac:.3f}  expected-under-independence={chance_same:.3f}  "
          f"(marginal={dict(marg)})  z={z:.2f}  n={len(pairs)}")
    assert abs(z) < 4.0, f"Possible leakage in {lang}: z={z:.2f} (observed {frac:.3f} vs expected {chance_same:.3f})"

print("Running leakage audits (expect ~0.50 for all languages)...")
for L in LANGS:
    leakage_audit(L, n=8000)


# ══════════════════════════════════════════════════════════════════════
# CELL 6 — Dataset
# ══════════════════════════════════════════════════════════════════════

class MultilingualPCFGDataset(IterableDataset):
    def __init__(self, langs, seed_offset=0, include_annots=True):
        self.langs = langs
        self.seed_offset = seed_offset
        # Training never touches annots (train_model only uses `ids`), but
        # with num_workers>0 every yielded item is pickled across a
        # process boundary regardless of what collate_fn does with it
        # afterward. A ~160-entry list of dicts per example, x batch_size,
        # x every step, turned out to be the actual bottleneck (GPU sat
        # at ~8% util / <1GB while a tiny model trained) -- not sampler
        # speed, not GPU compute. Set include_annots=False for training
        # loaders; evaluation still uses the full annotated sampler
        # directly (single-process, no IPC, so the cost there is fine).
        self.include_annots = include_annots

    def __iter__(self):
        worker_info = torch.utils.data.get_worker_info()
        wid = worker_info.id if worker_info else 0
        rng = random.Random(1000 * wid + self.seed_offset)
        while True:
            lang = rng.choice(self.langs)
            ids, annots, lang_out = sample_packed_sequence(lang, rng)
            yield torch.tensor(ids, dtype=torch.long), (annots if self.include_annots else None), lang_out

def collate(batch):
    ids = torch.stack([b[0] for b in batch])
    annots = [b[1] for b in batch]  # may be a list of Nones for training loaders
    langs = [b[2] for b in batch]
    return ids, annots, langs

def make_loader(langs, batch_size, seed_offset=0, include_annots=True):
    ds = MultilingualPCFGDataset(langs, seed_offset=seed_offset, include_annots=include_annots)
    return DataLoader(ds, batch_size=batch_size, collate_fn=collate,
                       num_workers=NUM_WORKERS, prefetch_factor=2,
                       persistent_workers=True, pin_memory=True)


# ══════════════════════════════════════════════════════════════════════
# CELL 7 — Model (small GPT-style decoder), parameterized by capacity
# ══════════════════════════════════════════════════════════════════════

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, max_len):
        super().__init__()
        self.n_head = n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd)
        self.proj = nn.Linear(n_embd, n_embd)
        mask = torch.tril(torch.ones(max_len, max_len)).view(1, 1, max_len, max_len)
        self.register_buffer('mask', mask)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=2)
        hd = C // self.n_head
        q = q.view(B, T, self.n_head, hd).transpose(1, 2)
        k = k.view(B, T, self.n_head, hd).transpose(1, 2)
        v = v.view(B, T, self.n_head, hd).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(hd)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        out = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out)

class Block(nn.Module):
    def __init__(self, n_embd, n_head, max_len):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, max_len)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.GELU(), nn.Linear(4 * n_embd, n_embd))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class TinyGPT(nn.Module):
    def __init__(self, vocab_size, n_embd, n_layer, n_head, max_len):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Parameter(torch.zeros(1, max_len, n_embd))
        self.blocks = nn.ModuleList([Block(n_embd, n_head, max_len) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
        self.max_len = max_len

    def forward(self, idx):
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pos_emb[:, :T, :]
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        return self.head(x)

    def num_params(self):
        return sum(p.numel() for p in self.parameters())

# Capacity grid: (n_layer, n_embd, n_head). Widen/narrow to taste --
# the point is to bracket the predicted N* crossovers, not to match any
# particular real-model scale.
# Capacity grid: (n_layer, n_embd, n_head). Widen/narrow to taste --
# the point is to bracket the predicted N* crossovers, not to match any
# particular real-model scale. Kept deliberately modest at the top end --
# attention is O(T^2) in MAX_LEN and cost scales worse than linearly with
# depth x width jointly, so the last point or two can dominate total
# wall-clock time even though it's "one more entry in a list."
CAPACITY_GRID = [
    (2,  64,  2),
    (2, 128,  4),
    (4, 128,  4),
    (4, 256,  8),
    (6, 256,  8),
]


# ══════════════════════════════════════════════════════════════════════
# CELL 8 — Training loop
# ══════════════════════════════════════════════════════════════════════

def train_model(n_layer, n_embd, n_head, langs, steps=3000, batch_size=256,
                 lr=6e-4, seed=0, log_every=500, use_dataloader=True):
    torch.manual_seed(seed)
    torch.backends.cudnn.benchmark = True  # fixed shapes every step (MAX_LEN constant) -- let cuDNN autotune once and reuse
    use_amp = device == "cuda"
    model = TinyGPT(VOCAB_SIZE, n_embd, n_layer, n_head, MAX_LEN).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    pad_id = STOI[PAD]

    if use_dataloader:
        loader = make_loader(langs, batch_size, seed_offset=seed, include_annots=False)
        it = iter(loader)
        def get_batch():
            ids, annots, batch_langs = next(it)
            return ids
    else:
        # Diagnostic path: bypass DataLoader/multiprocessing entirely --
        # sample directly in the main process, single-threaded. If a
        # stall that reproduces with the DataLoader disappears here, the
        # DataLoader/worker machinery is the cause, not the sampler or
        # the model/GPU side.
        rng = random.Random(seed)
        def get_batch():
            batch = [sample_packed_sequence(rng.choice(langs), rng)[0] for _ in range(batch_size)]
            return torch.tensor(batch, dtype=torch.long)

    model.train()
    t0 = time.time()
    for step in range(steps):
        ids = get_batch()
        ids = ids.to(device, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_amp):
            logits = model(ids[:, :-1])
            targets = ids[:, 1:].clone()
            targets[targets == pad_id] = -100
            loss = F.cross_entropy(logits.reshape(-1, VOCAB_SIZE), targets.reshape(-1), ignore_index=-100)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        if step % log_every == 0:
            print(f"  [N={model.num_params():,}] step {step:5d}  loss {loss.item():.4f}  "
                  f"({time.time()-t0:.0f}s elapsed)")
    return model


# ══════════════════════════════════════════════════════════════════════
# CELL 9 — Evaluation: restricted-vocab feature accuracy + Wilson CI
# ══════════════════════════════════════════════════════════════════════

def wilson_ci(successes, n, z=1.96):
    if n == 0:
        return (float('nan'), float('nan'))
    p = successes / n
    denom = 1 + z**2 / n
    centre = p + z**2 / (2 * n)
    adj = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))
    return ((centre - adj) / denom, (centre + adj) / denom)

@torch.no_grad()
def feature_accuracy_at_subcomp(model, lang, subcomp, n_eval=4000, batch_size=256, seed=999):
    """
    Restricted-vocab feature accuracy for a gender/class-bearing subcomp
    ('adj_agree', 'adj_agree_pre', 'v_agree', 'v_agree_obj'). Groups all
    valid tokens at that position by their target FEATURE (gender/class)
    and takes argmax over feature groups, not over the full vocabulary.

    Vectorized: matching (batch_row, position) pairs are gathered per
    eval batch and scored in ONE logsumexp call per feature group,
    instead of one torch call + .item() PER MATCHING POSITION. The
    per-position version forces a GPU-CPU sync at every single match
    (hundreds per batch) -- that's what was actually behind the GPU-util
    cliff right after training ended and eval started, not a real
    slowdown in the model or sampler.
    """
    model.eval()
    genders = genders_for(lang)
    if subcomp in ('adj_agree', 'adj_agree_pre'):
        groups = {g: [STOI[t] for t in ADJS[lang][g]] for g in genders}
    elif subcomp in ('v_agree', 'v_agree_obj'):
        spec = LANG_SPEC[lang]
        if spec['verb_agree'] != ('number', 'gender'):
            return None  # this subcomp doesn't carry a gender feature for this language
        groups = {}
        for g in genders:
            toks = []
            for n in ('SG', 'PL'):
                toks += VTR[lang][(g, n)] + VINTR[lang][(g, n)]
            groups[g] = [STOI[t] for t in toks]
    else:
        raise ValueError(subcomp)

    group_names = list(groups.keys())
    gender_to_idx = {g: i for i, g in enumerate(group_names)}
    # Index tensors built once, reused every batch -- no re-allocation per match.
    group_idx_tensors = [torch.tensor(groups[g], dtype=torch.long, device=device) for g in group_names]

    rng = random.Random(seed)
    score_chunks, label_chunks = [], []
    remaining = n_eval
    while remaining > 0:
        bs = min(batch_size, remaining)
        batch_ids, batch_ann = [], []
        for _ in range(bs):
            ids, ann, _ = sample_packed_sequence(lang, rng)
            batch_ids.append(ids)
            batch_ann.append(ann)
        ids_t = torch.tensor(batch_ids, dtype=torch.long, device=device)
        logits = model(ids_t[:, :-1])
        log_probs = F.log_softmax(logits, dim=-1)  # (B, T-1, V)

        # Cheap Python-side scan to find WHICH (row, pred_pos) pairs match --
        # this part is fine (dict lookups, no GPU calls). The GPU work is
        # deferred until every match in this batch is collected.
        rows, cols, labels = [], [], []
        for b, ann in enumerate(batch_ann):
            for pos, a in enumerate(ann):
                if a['subcomp'] != subcomp:
                    continue
                true_g = a.get('target_gender')
                if true_g is None:
                    continue
                pred_pos = pos - 1
                if pred_pos < 0:
                    continue
                rows.append(b)
                cols.append(pred_pos)
                labels.append(gender_to_idx[true_g])

        if rows:
            rows_t = torch.tensor(rows, device=device)
            cols_t = torch.tensor(cols, device=device)
            gathered = log_probs[rows_t, cols_t]  # (M, V) -- one gather, not M gathers
            # One logsumexp call PER GROUP (2 or 4 total), not per match.
            scores = torch.stack(
                [torch.logsumexp(gathered[:, idx], dim=1) for idx in group_idx_tensors], dim=1
            )  # (M, n_groups)
            score_chunks.append(scores.cpu())          # single sync per batch, not per match
            label_chunks.append(torch.tensor(labels))
        remaining -= bs

    if not score_chunks:
        return None
    all_scores = torch.cat(score_chunks, dim=0)
    all_labels = torch.cat(label_chunks, dim=0)
    preds = all_scores.argmax(dim=1)
    n_total = all_labels.numel()
    n_correct = (preds == all_labels).sum().item()
    acc = n_correct / n_total
    lo, hi = wilson_ci(n_correct, n_total)
    return dict(lang=lang, subcomp=subcomp, n=n_total, acc=acc, ci_lo=lo, ci_hi=hi)


def evaluate_capacity_point(model, langs, subcomps=('adj_agree', 'adj_agree_pre', 'v_agree', 'v_agree_obj')):
    results = []
    for lang in langs:
        for sc in subcomps:
            r = feature_accuracy_at_subcomp(model, lang, sc)
            if r is not None:
                results.append(r)
    return results


# ══════════════════════════════════════════════════════════════════════
# CELL 10 — Full sweep: train at every capacity point, every seed, evaluate
# ══════════════════════════════════════════════════════════════════════

def run_sweep(langs=LANGS, seeds=(0, 1), steps=3000,
              checkpoint_path='./v9_sweep_results.json',
              resume=True):
    """
    Returns a flat list of dicts: one row per (capacity, seed, lang, subcomp).
    Run this, then feed the output to extract_N_star below.
    Per the learnings file: always run >=2 seeds before trusting any
    single striking number -- this is enforced by the default `seeds`.

    Checkpointing: after EVERY (capacity, seed) point finishes, the
    accumulated rows are written to `checkpoint_path`. If `resume=True`
    and that file already exists, already-completed (capacity, seed)
    combinations are loaded and skipped rather than retrained -- so a
    killed or interrupted sweep can be restarted from the same command
    without losing finished runs. Completion is judged per (n_layer,
    n_embd, n_head, seed) tuple, not per individual eval row, since a
    given capacity/seed point either finished training+eval or it didn't.
    """
    rows = []
    done = set()
    if resume and os.path.exists(checkpoint_path):
        with open(checkpoint_path) as f:
            rows = json.load(f)
        done = {(r['n_layer'], r['n_embd'], r['n_head'], r['seed']) for r in rows}
        print(f"Resuming from checkpoint: {len(rows)} rows already recorded, "
              f"{len(done)} (capacity, seed) points already done.")

    for (n_layer, n_embd, n_head) in CAPACITY_GRID:
        for seed in seeds:
            key = (n_layer, n_embd, n_head, seed)
            if key in done:
                print(f"\n=== capacity (L={n_layer}, D={n_embd}, H={n_head})  seed={seed}  "
                      f"-- SKIPPED, already in checkpoint ===")
                continue
            print(f"\n=== capacity (L={n_layer}, D={n_embd}, H={n_head})  seed={seed} ===")
            model = train_model(n_layer, n_embd, n_head, langs, steps=steps, seed=seed)
            n_params = model.num_params()
            evals = evaluate_capacity_point(model, langs)
            for e in evals:
                e.update(n_params=n_params, n_layer=n_layer, n_embd=n_embd,
                          n_head=n_head, seed=seed)
                rows.append(e)
            print(f"  -- eval @ N={n_params:,}, seed={seed} --")
            for e in sorted(evals, key=lambda r: (r['subcomp'], r['lang'])):
                print(f"     {e['lang']:6s} {e['subcomp']:14s} acc={e['acc']:.3f} "
                      f"[{e['ci_lo']:.3f},{e['ci_hi']:.3f}]  n={e['n']}")
            del model
            torch.cuda.empty_cache()

            # Checkpoint immediately -- a run killed mid-sweep (as just
            # happened) should never cost you more than the ONE
            # (capacity, seed) point that was in flight.
            with open(checkpoint_path, 'w') as f:
                json.dump(rows, f, indent=2)
            print(f"  (checkpointed {len(rows)} total rows to {checkpoint_path})")

    return rows


# ══════════════════════════════════════════════════════════════════════
# CELL 11 — N* extraction and twin-pair comparison
# ══════════════════════════════════════════════════════════════════════

def crossover_capacity(rows, lang, subcomp, threshold=0.75):
    """
    Empirical N* proxy: smallest evaluated capacity (in params) at which
    mean accuracy across seeds first exceeds `threshold`. Returns
    (n_params, accs_at_each_capacity) or (None, accs) if never crossed
    within the swept grid (consistent with the theory's own N*->infinity
    edge cases -- report this as a real result, not a failure).
    """
    by_capacity = defaultdict(list)
    for r in rows:
        if r['lang'] == lang and r['subcomp'] == subcomp:
            by_capacity[r['n_params']].append(r['acc'])
    if not by_capacity:
        return None, {}
    means = {n: sum(v) / len(v) for n, v in sorted(by_capacity.items())}
    for n, m in means.items():
        if m >= threshold:
            return n, means
    return None, means

def summarize_twin_pairs(rows, subcomp='adj_agree', threshold=0.75):
    print(f"\n--- N* summary at subcomp='{subcomp}', threshold={threshold} ---")
    for base, twin, axis in TWIN_PAIRS:
        n_base, means_base = crossover_capacity(rows, base, subcomp, threshold)
        n_twin, means_twin = crossover_capacity(rows, twin, subcomp, threshold)
        print(f"\n[{axis}]")
        print(f"  {base:6s} N* = {n_base if n_base else '>max swept'}   "
              f"accs by capacity: {['%.2f' % v for v in means_base.values()]}")
        print(f"  {twin:6s} N* = {n_twin if n_twin else '>max swept'}   "
              f"accs by capacity: {['%.2f' % v for v in means_twin.values()]}")
        if n_base and n_twin:
            direction = "twin separates LATER (higher N*)" if n_twin > n_base else \
                        "twin separates EARLIER (lower N*)" if n_twin < n_base else \
                        "same N*"
            print(f"  -> {direction}")


# ══════════════════════════════════════════════════════════════════════
# CELL 12 — Entry point
# ══════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    # Sanity: print a handful of sampled sentences per language before
    # spending any compute, so a grammar bug is caught immediately.
    rng = random.Random(42)
    for L in LANGS:
        ids, ann, _ = sample_packed_sequence(L, rng)
        toks = [ITOS[i] for i in ids if ITOS[i] not in (PAD,)]
        print(f"\n[{L}] {' '.join(toks[:24])} ...")

    print("\nStarting capacity x seed sweep (checkpointed after every point --")
    print("safe to Ctrl+C and rerun this script to resume).")
    rows = run_sweep(seeds=(0, 1), steps=3000)

    print(f"\nDone. {len(rows)} total rows in ./v9_sweep_results.json")

    summarize_twin_pairs(rows, subcomp='adj_agree', threshold=0.75)
    summarize_twin_pairs(rows, subcomp='adj_agree_pre', threshold=0.75)
    summarize_twin_pairs(rows, subcomp='v_agree_obj', threshold=0.75)

Device: cuda  |  NUM_WORKERS: 16  (cpu_count=40)
Gender tables built and Hamming distances verified against spec.
Vocabulary: 533 tokens  |  Languages: ['A', 'B', 'C', 'Ctau', 'D', 'Dlo', 'Dpost', 'E']  |  Shared noun stems: 40
Running leakage audits (expect ~0.50 for all languages)...
[A] observed P(match)=0.504  expected-under-independence=0.511  (marginal={'F': 61048, 'M': 45314})  z=-1.31  n=8000
[B] observed P(match)=0.515  expected-under-independence=0.511  (marginal={'M': 57198, 'F': 76916})  z=0.79  n=8000
[C] observed P(match)=0.496  expected-under-independence=0.501  (marginal={'F': 50277, 'M': 56038})  z=-1.00  n=8000
[Ctau] observed P(match)=0.508  expected-under-independence=0.501  (marginal={'F': 50446, 'M': 55869})  z=1.29  n=8000
[D] observed P(match)=0.513  expected-under-independence=0.511  (marginal={'M': 61349, 'F': 45622})  z=0.48  n=8000
[Dlo] observed P(match)=0.496  expected-under-independence=0.501  (marginal={'F': 55799, 'M': 51172})  z=-0.84  n=8000
[Dpost] o

In [2]:
model = train_model(2, 64, 2, LANGS, steps=3000, use_dataloader=False)

  [N=178,560] step     0  loss 6.4493  (0s elapsed)
  [N=178,560] step   500  loss 2.2277  (43s elapsed)
  [N=178,560] step  1000  loss 2.0706  (85s elapsed)
  [N=178,560] step  1500  loss 1.9243  (128s elapsed)
  [N=178,560] step  2000  loss 1.8815  (171s elapsed)
  [N=178,560] step  2500  loss 1.8694  (214s elapsed)


In [1]:
#!/usr/bin/env python3
"""
Multilingual PCFG toy model — v9: shared vocabulary + twin-pair design
=======================================================================

WHY THIS EXISTS (context from the v8 series):

v8's languages (A-E) had fully disjoint per-language noun vocabularies.
That gave each language its own clean embedding subspace, which means
there was never any genuine representational competition -- i.e. eps
was structurally impossible to create, no matter how the grammar was
tuned. Every attempt to force an eps signal out of v8's setup (see
L_E's adj_agree_pre) either measured "this sub-computation is provably
unsolvable here" (N* -> infinity, a correct null result, not a
crossover) or a noisy activation-similarity artifact (Pearson tracking
generic magnitude convergence, not circuit sharing -- see learnings).

v9's fix has two parts:

  1. SHARED VOCABULARY. All languages draw nouns from ONE shared pool
     of 40 stems. Each language has its OWN gender-assignment table
     over those stems. The same noun embedding must therefore serve
     contradictory gender readouts depending on which language's
     context it appears in. That contradiction, under finite capacity,
     IS eps: a genuine representational bottleneck, not a simulated
     one.

  2. TWIN-PAIR DESIGN. A rich "pool" of 5 naturalistic languages
     (A/B/C/D/E) creates real capacity pressure (many overlapping
     sub-computations competing for shared weights -- single-task toy
     models don't create this, per the learnings file). But comparing
     naturalistic languages pairwise confounds every axis at once
     (word order + case + verb target + gender distance all differ
     simultaneously), so you can't attribute an observed N* to any one
     term in the formula.

     The fix: add TWINS -- languages identical to a pool language in
     every respect except ONE toggled axis:
       - D  vs Dlo   : identical structure, hamming 34 vs 6  -> isolates eps
       - C  vs Ctau  : identical structure, det marks gender or not -> isolates tau
       - D  vs Dpost : identical structure, adj pre- vs post-nominal -> isolates
                       STRUCTURAL eps (does losing noun-context at the
                       agreement site add cost beyond lexical distance alone)

     Twins ADD to the pool's total sub-computation count (more capacity
     pressure) while still allowing clean, single-axis comparisons
     (identifiability). This is not "more realism" -- every twin exists
     because it isolates a term in the N* formula, per the standing
     principle: don't add machinery unless it changes a number in an
     actual formula.

THE EIGHT LANGUAGES (see LANG_SPEC below for exact toggles):

  A     anchor.        SVO, post-N, DET+gender(2),  no case, V:number,        hamming=0
  B     near-anchor.   SVO, post-N, DET+gender+num, no case, V:number+gender, hamming=6
  C     tau_lo probe.  SOV, post-N, DET invariant,  +case,  V:none (absent),  hamming=20
  Ctau  tau_hi twin of C (DET now marks gender). Everything else = C.        hamming=20
  D     eps_hi, structural. SOV, PRE-N, DET+gender, +case, V:number+gender(OBJECT), hamming=34
  Dlo   eps_lo twin of D (hamming 6 instead of 34). Everything else = D.
  Dpost structural twin of D (post-N instead of pre-N). Everything else = D.  hamming=34
  E     system-mismatch. SVO, post-N, DET+CLASS(4-way), no case, V:number+class, hamming=N/A
        (E's 4-way class system is INCOMMENSURABLE with everyone else's
        2-way gender system -- the deepest eps: not just "different
        assignment" but "different dimensionality of category system".)

MEASUREMENT (per the learnings file -- apply these, don't re-derive them):
  - Behavioral accuracy is the primary signal, not activation similarity.
    Pearson/CKA-style metrics are known to be unreliable here.
  - Always use RESTRICTED-VOCAB FEATURE accuracy (sum log-probs over all
    tokens belonging to a gender/class, argmax over the small number of
    FEATURES, not over the full vocabulary) -- full-vocab top-1 caps well
    below 1.0 even with perfect feature reading, because lemma choice
    within the correct feature is uniform noise.
  - Always report Wilson CIs; always check per-branch sample sizes.
  - Run every capacity point at >=2 seeds before trusting a single number.
  - Ground truth for gender is a per-(language, noun-stem) property (a
    lookup table), NOT sampled independently per sentence -- this is the
    core difference from v8's grammar and is what actually creates eps.

Cell boundaries marked with # ====== CELL N ======. Paste each cell into
a notebook, or run the whole file as a script.
"""

# ══════════════════════════════════════════════════════════════════════
# CELL 1 — Setup
# ══════════════════════════════════════════════════════════════════════

import os
GPU_ID = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

import math, random, time, json, itertools
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Tuple, Set, Any
from collections import defaultdict, Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import IterableDataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = 16
print(f"Device: {device}  |  NUM_WORKERS: {NUM_WORKERS}  (cpu_count={os.cpu_count()})")


# ══════════════════════════════════════════════════════════════════════
# CELL 2 — Language specification table (every axis is an explicit toggle)
# ══════════════════════════════════════════════════════════════════════

N_NOUN_STEMS  = 160    # SHARED across all languages (this is the eps mechanism).
                        # Was 40 in the first run -- with 8 languages that's only
                        # 320 (lang, noun) -> gender entries, which a transformer
                        # solves as a trivial lookup table at EVERY swept capacity
                        # (confirmed empirically: acc=1.000 even at 178K params).
                        # 160 stems raises this to 1280 entries and, combined with
                        # the embedding-width cap below, is meant to actually make
                        # memorizing the full per-language table costly relative
                        # to sharing.
N_ADJ_LEMMAS  = 8
N_VTR_LEMMAS  = 6
N_VINTR_LEMMAS = 4
N_PREPS       = 4

GENDER_SYSTEMS = {
    2: ['M', 'F'],
    4: ['C1', 'C2', 'C3', 'C4'],
}

# hamming: for 2-gender languages, number of the 40 shared stems whose
# gender differs from the anchor (A)'s assignment. 'incommensurable' for
# E, whose 4-way system has no shared axis to measure Hamming distance
# against in the first place.
LANG_SPEC: Dict[str, Dict[str, Any]] = {
    'A': dict(anchor=True,  word_order='SVO', adj_pos='post',
              gender_system=2, det_marks_gender=True,  det_fuses_number=False,
              case_marking=False, verb_agree=('number',),          verb_agree_target='subject',
              rel_agrees=True,  hamming=0),

    'B': dict(anchor=False, word_order='SVO', adj_pos='post',
              gender_system=2, det_marks_gender=True,  det_fuses_number=True,
              case_marking=False, verb_agree=('number', 'gender'), verb_agree_target='subject',
              rel_agrees=True,  hamming=6),

    'C': dict(anchor=False, word_order='SOV', adj_pos='post',
              gender_system=2, det_marks_gender=False, det_fuses_number=False,
              case_marking=True,  verb_agree=(),                   verb_agree_target=None,
              rel_agrees=False, hamming=20),

    'Ctau': dict(anchor=False, word_order='SOV', adj_pos='post',           # = C, tau flipped
              gender_system=2, det_marks_gender=True,  det_fuses_number=False,
              case_marking=True,  verb_agree=(),                   verb_agree_target=None,
              rel_agrees=False, hamming=20),

    'D': dict(anchor=False, word_order='SOV', adj_pos='pre',
              gender_system=2, det_marks_gender=True,  det_fuses_number=False,
              case_marking=True,  verb_agree=('number', 'gender'), verb_agree_target='object',
              rel_agrees=True,  hamming=34),

    'Dlo': dict(anchor=False, word_order='SOV', adj_pos='pre',            # = D, eps flipped
              gender_system=2, det_marks_gender=True,  det_fuses_number=False,
              case_marking=True,  verb_agree=('number', 'gender'), verb_agree_target='object',
              rel_agrees=True,  hamming=6),

    'Dpost': dict(anchor=False, word_order='SOV', adj_pos='post',         # = D, structure flipped
              gender_system=2, det_marks_gender=True,  det_fuses_number=False,
              case_marking=True,  verb_agree=('number', 'gender'), verb_agree_target='object',
              rel_agrees=True,  hamming=34),

    'E': dict(anchor=False, word_order='SVO', adj_pos='post',
              gender_system=4, det_marks_gender=True,  det_fuses_number=False,
              case_marking=False, verb_agree=('number', 'gender'), verb_agree_target='subject',
              rel_agrees=True,  hamming='incommensurable'),
}
LANGS = list(LANG_SPEC.keys())

# Named comparison pairs this design exists to support. Each entry is
# (base, twin, axis_isolated) -- used later by the N* extraction cell.
TWIN_PAIRS = [
    ('D', 'Dlo',   'epsilon (lexical gender distance: hamming 34 vs 6)'),
    ('C', 'Ctau',  'tau (DET marks gender: no vs yes)'),
    ('D', 'Dpost', 'structural epsilon (adj position: pre- vs post-nominal)'),
    ('A', 'B',     'epsilon, low end (hamming 0 vs 6), naturalistic pool'),
    ('A', 'E',     'epsilon, system-mismatch (2-way vs 4-way gender)'),
]

def genders_for(lang):
    return GENDER_SYSTEMS[LANG_SPEC[lang]['gender_system']]


# ── Gender assignment tables: the core eps mechanism ──
# Ground truth gender is a property of (language, noun stem) -- a lookup
# table, not sampled per sentence. The anchor's table is built once; every
# other 2-gender language's table starts as a COPY of the anchor's and
# flips `hamming` stems (chosen deterministically per language). E's
# 4-way table is independent and uses no relation to the anchor's axis.

NOUN_STEMS = [f"n{idx:02d}" for idx in range(N_NOUN_STEMS)]

def build_anchor_table(seed=0):
    rng = random.Random(seed)
    return {stem: rng.choice(['M', 'F']) for stem in NOUN_STEMS}

ANCHOR_TABLE = build_anchor_table(seed=0)

def build_gender_table(lang):
    spec = LANG_SPEC[lang]
    if spec['gender_system'] == 4:
        rng = random.Random(hash(('gender4', lang)) & 0xFFFFFFFF)
        return {stem: rng.choice(GENDER_SYSTEMS[4]) for stem in NOUN_STEMS}
    if spec.get('anchor'):
        return dict(ANCHOR_TABLE)
    h = spec['hamming']
    rng = random.Random(hash(('flip', lang)) & 0xFFFFFFFF)
    flip_stems = set(rng.sample(NOUN_STEMS, h))
    table = dict(ANCHOR_TABLE)
    for stem in flip_stems:
        table[stem] = 'F' if table[stem] == 'M' else 'M'
    return table

GENDER_TABLE: Dict[str, Dict[str, str]] = {L: build_gender_table(L) for L in LANGS}

# Sanity check: verify realized Hamming distance matches spec (twins must
# match their base language exactly on every stem except the intended flip
# count, or the "identical except one axis" claim is false).
for L in LANGS:
    spec = LANG_SPEC[L]
    if spec['gender_system'] == 2 and not spec.get('anchor'):
        realized = sum(1 for s in NOUN_STEMS if GENDER_TABLE[L][s] != ANCHOR_TABLE[s])
        assert realized == spec['hamming'], (L, realized, spec['hamming'])
print("Gender tables built and Hamming distances verified against spec.")


# ══════════════════════════════════════════════════════════════════════
# CELL 3 — Vocabulary construction (generic over LANG_SPEC)
# ══════════════════════════════════════════════════════════════════════
#
# Two disjoint SHARED noun pools:
#   BARE_NOUNS  -- used by non-case-marking languages (A, B, E). A bare
#                  stem token (e.g. "n07") is identical across A/B/E, so
#                  its embedding must serve THREE different gender/class
#                  readouts depending on language context.
#   CASE_NOUNS  -- used by case-marking languages (C, Ctau, D, Dlo,
#                  Dpost). Each stem appears as stem+"_NOM" / stem+"_ACC",
#                  shared identically across that subset of languages.
# These two pools never overlap, so representational competition on a
# given noun token is scoped to the languages that actually use that
# surface form -- exactly mirroring how gender clashes in real languages
# are scoped to languages sharing a cognate or loanword, not universal.

DOT = '.'
BOS, EOS, PAD = '<BOS>', '<EOS>', '<PAD>'
LANG_TOK = {L: f'<L{L}>' for L in LANGS}

BARE_NOUNS = list(NOUN_STEMS)
CASE_NOUNS = {(stem, case): f"{stem}_{case}" for stem in NOUN_STEMS for case in ('NOM', 'ACC')}

def noun_surface(lang, stem, role):
    """role: 'subj' or 'obj'. Returns the surface token for this stem in this language."""
    if LANG_SPEC[lang]['case_marking']:
        case = 'NOM' if role == 'subj' else 'ACC'
        return CASE_NOUNS[(stem, case)]
    return stem  # bare, shared across non-case languages

# ── Per-language functional morphology (DET, NUM, ADJ, V, REL, PREP, CONJ) ──
DETS: Dict[str, Any] = {}
NUMS: Dict[str, Dict[str, str]] = {}
ADJS: Dict[str, Dict[str, List[str]]] = {}
VTR: Dict[str, Any] = {}
VINTR: Dict[str, Any] = {}
RPROS: Dict[str, Any] = {}
PREPS: Dict[str, List[str]] = {}
CONJS: Dict[str, str] = {}

for L in LANGS:
    spec = LANG_SPEC[L]
    genders = genders_for(L)

    # DET
    if spec['det_marks_gender'] and spec['det_fuses_number']:
        DETS[L] = {(g, n): f"dt_{L}_{g}_{n}" for g in genders for n in ('SG', 'PL')}
    elif spec['det_marks_gender']:
        DETS[L] = {g: f"dt_{L}_{g}" for g in genders}
    else:
        DETS[L] = f"dt_{L}"  # single invariant token

    # NUM particle (absent if number is fused into DET)
    if not spec['det_fuses_number']:
        NUMS[L] = {'SG': f"ns_{L}", 'PL': f"np_{L}"}

    # ADJ: one lemma set, N_ADJ_LEMMAS lemmas, inflected per gender/class
    ADJS[L] = {g: [f"adj_{L}_{g}_{i}" for i in range(N_ADJ_LEMMAS)] for g in genders}

    # Verb agreement features
    feats = spec['verb_agree']
    def make_verb_forms(n_lemmas, L=L, genders=genders, feats=feats):
        if feats == ():
            return {'__inv__': [f"v_{L}_{i}" for i in range(n_lemmas)]}
        if feats == ('number',):
            return {n: [f"v_{L}_{n}_{i}" for i in range(n_lemmas)] for n in ('SG', 'PL')}
        if feats == ('number', 'gender'):
            return {(g, n): [f"v_{L}_{g}_{n}_{i}" for i in range(n_lemmas)]
                    for g in genders for n in ('SG', 'PL')}
        raise ValueError(feats)
    VTR[L]   = make_verb_forms(N_VTR_LEMMAS)
    VINTR[L] = make_verb_forms(N_VINTR_LEMMAS)

    # Relative pronoun
    RPROS[L] = ({g: f"rp_{L}_{g}" for g in genders} if spec['rel_agrees'] else f"rp_{L}")

    PREPS[L] = [f"p_{L}{i}" for i in range(N_PREPS)]
    CONJS[L] = f"cj_{L}"

# ── Build full vocabulary ──
ALL_TOKENS: Set[str] = set()
ALL_TOKENS.update(BARE_NOUNS)
ALL_TOKENS.update(CASE_NOUNS.values())
ALL_TOKENS.update([DOT, BOS, EOS, PAD])
ALL_TOKENS.update(LANG_TOK.values())

for L in LANGS:
    d = DETS[L]
    ALL_TOKENS.update(d.values() if isinstance(d, dict) else [d])
    if L in NUMS:
        ALL_TOKENS.update(NUMS[L].values())
    for g in genders_for(L):
        ALL_TOKENS.update(ADJS[L][g])
    for forms in VTR[L].values():
        ALL_TOKENS.update(forms)
    for forms in VINTR[L].values():
        ALL_TOKENS.update(forms)
    r = RPROS[L]
    ALL_TOKENS.update(r.values() if isinstance(r, dict) else [r])
    ALL_TOKENS.update(PREPS[L])
    ALL_TOKENS.add(CONJS[L])

VOCAB = sorted(ALL_TOKENS)
VOCAB_SIZE = len(VOCAB)
STOI = {w: i for i, w in enumerate(VOCAB)}
ITOS = {i: w for w, i in STOI.items()}
MAX_LEN = 160

print(f"Vocabulary: {VOCAB_SIZE} tokens  |  Languages: {LANGS}  |  Shared noun stems: {N_NOUN_STEMS}")


# ══════════════════════════════════════════════════════════════════════
# CELL 4 — Generic PCFG sampler, driven entirely by LANG_SPEC
# ══════════════════════════════════════════════════════════════════════
#
# Grammar (shared non-terminal skeleton; language-specific realization):
#   S      -> CLAUSE | CLAUSE CONJ CLAUSE
#   CLAUSE -> depends on word_order:
#               SVO: NP_subj VP(-> V [NP_obj])
#               SOV: NP_subj [NP_obj] V
#   NP     -> DET_region N [ADJ] [PP] | DET_region N [ADJ] REL
#   PP     -> PREP NP
#   REL    -> RPRO V(agrees w/ head number)
#
# Ground-truth gender/class for a noun stem is a LOOKUP (GENDER_TABLE),
# not sampled per sentence -- this is what makes the same stem carry
# different gender information depending on which language is speaking.

P_ADJ  = 0.45
P_PP   = 0.15
P_REL  = 0.12
P_TRANSITIVE = 0.65
MAX_PP_DEPTH  = 1
MAX_REL_DEPTH = 1

@dataclass
class NPFeatures:
    gender: str
    number: str
    stem: str
    noun_pos: int = -1

def sample_np(lang, rng, role, pp_depth, rel_depth, emit):
    """role: 'subj' or 'obj'. emit(tok, subcomp, **extra) appends to the sequence."""
    spec = LANG_SPEC[lang]
    stem = rng.choice(NOUN_STEMS)
    gender = GENDER_TABLE[lang][stem]          # <-- ground truth, per-language lookup
    number = rng.choice(['SG', 'PL'])
    noun_tok = noun_surface(lang, stem, role)

    # ── DET region ──
    d = DETS[lang]
    if spec['det_marks_gender'] and spec['det_fuses_number']:
        emit(d[(gender, number)], 'det', gender=gender, number=number)
    elif spec['det_marks_gender']:
        emit(d[gender], 'det', gender=gender, number=number)
        emit(NUMS[lang][number], 'num', number=number)
    else:
        emit(d, 'det', gender=None, number=None)
        emit(NUMS[lang][number], 'num', number=number)

    r = rng.random()
    has_adj = r < P_ADJ
    has_pp  = (not has_adj) and r < P_ADJ + P_PP and pp_depth[0] < MAX_PP_DEPTH
    has_rel = (not has_adj) and (not has_pp) and r < P_ADJ + P_PP + P_REL and rel_depth[0] < MAX_REL_DEPTH

    def emit_adj():
        lemma_idx = rng.randrange(N_ADJ_LEMMAS)
        emit(ADJS[lang][gender][lemma_idx],
             'adj_agree_pre' if spec['adj_pos'] == 'pre' else 'adj_agree',
             target_gender=gender, lemma=lemma_idx, stem=stem)

    if spec['adj_pos'] == 'pre' and has_adj:
        emit_adj()

    noun_pos = len(pp_depth[1])  # pp_depth[1] is the running token list, see wrapper below
    emit(noun_tok, 'n_after_det', gender=gender, number=number, stem=stem)
    npf = NPFeatures(gender=gender, number=number, stem=stem, noun_pos=noun_pos)

    if spec['adj_pos'] == 'post' and has_adj:
        emit_adj()
        if rng.random() < P_PP and pp_depth[0] < MAX_PP_DEPTH:
            sample_pp(lang, rng, pp_depth, rel_depth, emit)
    elif has_pp:
        sample_pp(lang, rng, pp_depth, rel_depth, emit)
    elif has_rel:
        sample_rel(lang, rng, npf, pp_depth, rel_depth, emit)

    return npf

def sample_pp(lang, rng, pp_depth, rel_depth, emit):
    emit(rng.choice(PREPS[lang]), 'structural')
    pp_depth[0] += 1
    sample_np(lang, rng, 'obj', pp_depth, rel_depth, emit)
    pp_depth[0] -= 1

def sample_rel(lang, rng, head, pp_depth, rel_depth, emit):
    spec = LANG_SPEC[lang]
    r = RPROS[lang]
    emit(r[head.gender] if isinstance(r, dict) else r, 'rel_pro',
         target_gender=head.gender if isinstance(r, dict) else None)
    rel_depth[0] += 1
    # REL-internal verb agrees with the head noun's NUMBER only (this is a
    # relative-clause-internal fact, independent of whether the language's
    # MAIN-clause verb also marks gender/class -- so key into VINTR using
    # the same (feats -> key) mapping as pick_verb_form, just always with
    # the head's number rather than a freshly-drawn agreement target).
    forms = VINTR[lang]
    feats = spec['verb_agree']
    lemma_idx = rng.randrange(N_VINTR_LEMMAS)
    if feats == ():
        key = '__inv__'
    elif feats == ('number',):
        key = head.number
    else:  # ('number', 'gender')
        key = (head.gender, head.number)
    tok = forms[key][lemma_idx]
    emit(tok, 'v_agree_rel', target_number=head.number, head_stem=head.stem)
    rel_depth[0] -= 1

def pick_verb_form(lang, agree_npf, rng, transitive):
    """agree_npf: the NPFeatures whose number/gender the verb must agree with (subject or object)."""
    spec = LANG_SPEC[lang]
    forms_table = VTR[lang] if transitive else VINTR[lang]
    n_lemmas = N_VTR_LEMMAS if transitive else N_VINTR_LEMMAS
    lemma_idx = rng.randrange(n_lemmas)
    feats = spec['verb_agree']
    if feats == ():
        key = '__inv__'
    elif feats == ('number',):
        key = agree_npf.number
    else:  # ('number', 'gender')
        key = (agree_npf.gender, agree_npf.number)
    tok = forms_table[key][lemma_idx]
    subcomp = 'v_agree' if spec['verb_agree_target'] != 'object' else 'v_agree_obj'
    return tok, subcomp, key

def sample_clause(lang, rng, pp_depth, rel_depth, emit):
    spec = LANG_SPEC[lang]

    if spec['word_order'] == 'SVO':
        subj = sample_np(lang, rng, 'subj', pp_depth, rel_depth, emit)
        transitive = rng.random() < P_TRANSITIVE
        agree_target = subj  # SVO languages here all agree with subject
        tok, subcomp, key = pick_verb_form(lang, agree_target, rng, transitive)
        emit(tok, subcomp, target_number=agree_target.number,
             target_gender=agree_target.gender if isinstance(key, tuple) else None,
             agree_source='subject')
        if transitive:
            sample_np(lang, rng, 'obj', pp_depth, rel_depth, emit)

    else:  # SOV
        subj = sample_np(lang, rng, 'subj', pp_depth, rel_depth, emit)
        transitive = rng.random() < P_TRANSITIVE
        obj = None
        if transitive:
            obj = sample_np(lang, rng, 'obj', pp_depth, rel_depth, emit)
        agree_target = subj
        if spec['verb_agree_target'] == 'object' and obj is not None:
            agree_target = obj
        tok, subcomp, key = pick_verb_form(lang, agree_target, rng, transitive)
        emit(tok, subcomp, target_number=agree_target.number,
             target_gender=agree_target.gender if isinstance(key, tuple) else None,
             agree_source=('object' if agree_target is obj else 'subject'))

    emit(DOT, 'structural')

def sample_sentence(lang, rng):
    tokens, annots = [], []
    def emit(tok, subcomp, **extra):
        tokens.append(tok)
        ann = {'subcomp': subcomp}
        ann.update(extra)
        annots.append(ann)
    pp_depth = [0, tokens]   # pp_depth[0]=depth counter, pp_depth[1]=running tokens (for noun_pos)
    rel_depth = [0]
    sample_clause(lang, rng, pp_depth, rel_depth, emit)
    if rng.random() < 0.35:
        emit(CONJS[lang], 'structural')
        sample_clause(lang, rng, pp_depth, rel_depth, emit)
    return tokens, annots

def sample_packed_sequence(lang, rng, max_len=MAX_LEN):
    tokens = [BOS, LANG_TOK[lang]]
    annots = [{'subcomp': 'bos'}, {'subcomp': 'lang_id'}]
    while len(tokens) < max_len - 1:
        s_tok, s_ann = sample_sentence(lang, rng)
        if len(tokens) + len(s_tok) + 1 > max_len:
            break
        tokens.extend(s_tok)
        annots.extend(s_ann)
    tokens.append(EOS)
    annots.append({'subcomp': 'eos'})
    pad_len = max_len - len(tokens)
    tokens.extend([PAD] * pad_len)
    annots.extend([{'subcomp': 'pad'}] * pad_len)
    ids = [STOI[t] for t in tokens]
    return ids, annots, lang


# ══════════════════════════════════════════════════════════════════════
# CELL 5 — Leakage / independence audit
# ══════════════════════════════════════════════════════════════════════
#
# Two things must be verified before trusting any accuracy number:
#  (a) For languages with adj_pos == 'pre' AND det_marks_gender == False
#      -- there are none in this spec (that was a known dead end from
#      v8's L_E; C is post-nominal specifically to avoid it) -- but we
#      check the general claim programmatically rather than by
#      inspection, in case the spec changes later.
#  (b) Gender/number are drawn independently per NP (via the lookup
#      table keyed only on the freshly-sampled stem), so nothing
#      upstream in the sequence should predict a downstream NP's gender
#      beyond what the language's own DET/case system provides. Check:
#      within compound sentences, does clause 1's target_gender predict
#      clause 2's target_gender at its own adj_agree position, once you
#      condition on stem repetition being rare (40 stems, i.i.d. draws)?

def leakage_audit(lang, n=20000, seed=13):
    """
    Compares the observed same-gender rate across two successive ADJ
    targets against the CHANCE rate implied by that language's OWN
    empirical gender marginal -- not a fixed 0.5 (or 0.25 for E). This
    matters because a hamming-flip table need not preserve the anchor's
    M/F balance: e.g. flipping 34/40 stems can leave a language with an
    uneven split, which alone inflates same-gender match rate with no
    leakage involved. Flagging that as "leakage" against a hardcoded
    0.40-0.60 band would be a false positive -- diagnose before fixing.
    """
    rng = random.Random(seed)
    pairs, all_genders = [], []
    for _ in range(n):
        ids, ann, _ = sample_packed_sequence(lang, rng, max_len=MAX_LEN)
        gs = [a.get('target_gender') for a in ann
              if a['subcomp'] in ('adj_agree', 'adj_agree_pre') and a.get('target_gender')]
        all_genders.extend(gs)
        if len(gs) >= 2:
            pairs.append((gs[0], gs[1]))
    if len(pairs) < 200:
        print(f"[{lang}] too few multi-ADJ sequences ({len(pairs)}) for a leakage check -- skipping.")
        return
    marg = Counter(all_genders)
    total = sum(marg.values())
    chance_same = sum((c / total) ** 2 for c in marg.values())  # sum_g P(g)^2
    same = sum(1 for a, b in pairs if a == b)
    frac = same / len(pairs)
    se = math.sqrt(max(chance_same * (1 - chance_same), 1e-9) / len(pairs))
    z = (frac - chance_same) / se if se > 0 else 0.0
    print(f"[{lang}] observed P(match)={frac:.3f}  expected-under-independence={chance_same:.3f}  "
          f"(marginal={dict(marg)})  z={z:.2f}  n={len(pairs)}")
    assert abs(z) < 4.0, f"Possible leakage in {lang}: z={z:.2f} (observed {frac:.3f} vs expected {chance_same:.3f})"

print("Running leakage audits (expect ~0.50 for all languages)...")
for L in LANGS:
    leakage_audit(L, n=8000)


# ══════════════════════════════════════════════════════════════════════
# CELL 6 — Dataset
# ══════════════════════════════════════════════════════════════════════

class MultilingualPCFGDataset(IterableDataset):
    def __init__(self, langs, seed_offset=0, include_annots=True):
        self.langs = langs
        self.seed_offset = seed_offset
        # Training never touches annots (train_model only uses `ids`), but
        # with num_workers>0 every yielded item is pickled across a
        # process boundary regardless of what collate_fn does with it
        # afterward. A ~160-entry list of dicts per example, x batch_size,
        # x every step, turned out to be the actual bottleneck (GPU sat
        # at ~8% util / <1GB while a tiny model trained) -- not sampler
        # speed, not GPU compute. Set include_annots=False for training
        # loaders; evaluation still uses the full annotated sampler
        # directly (single-process, no IPC, so the cost there is fine).
        self.include_annots = include_annots

    def __iter__(self):
        worker_info = torch.utils.data.get_worker_info()
        wid = worker_info.id if worker_info else 0
        rng = random.Random(1000 * wid + self.seed_offset)
        while True:
            lang = rng.choice(self.langs)
            ids, annots, lang_out = sample_packed_sequence(lang, rng)
            yield torch.tensor(ids, dtype=torch.long), (annots if self.include_annots else None), lang_out

def collate(batch):
    ids = torch.stack([b[0] for b in batch])
    annots = [b[1] for b in batch]  # may be a list of Nones for training loaders
    langs = [b[2] for b in batch]
    return ids, annots, langs

def make_loader(langs, batch_size, seed_offset=0, include_annots=True):
    ds = MultilingualPCFGDataset(langs, seed_offset=seed_offset, include_annots=include_annots)
    return DataLoader(ds, batch_size=batch_size, collate_fn=collate,
                       num_workers=NUM_WORKERS, prefetch_factor=2,
                       persistent_workers=True, pin_memory=True)


# ══════════════════════════════════════════════════════════════════════
# CELL 7 — Model (small GPT-style decoder), parameterized by capacity
# ══════════════════════════════════════════════════════════════════════

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, max_len):
        super().__init__()
        self.n_head = n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd)
        self.proj = nn.Linear(n_embd, n_embd)
        mask = torch.tril(torch.ones(max_len, max_len)).view(1, 1, max_len, max_len)
        self.register_buffer('mask', mask)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=2)
        hd = C // self.n_head
        q = q.view(B, T, self.n_head, hd).transpose(1, 2)
        k = k.view(B, T, self.n_head, hd).transpose(1, 2)
        v = v.view(B, T, self.n_head, hd).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(hd)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        out = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out)

class Block(nn.Module):
    def __init__(self, n_embd, n_head, max_len):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, max_len)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.GELU(), nn.Linear(4 * n_embd, n_embd))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class TinyGPT(nn.Module):
    """
    n_embd_tok is the TOKEN EMBEDDING dimension -- the channel a
    (language, noun-identity) -> gender lookup has to be smuggled
    through. n_embd is the RESIDUAL/attention/MLP width everything else
    runs in. These were the same number in the first run, which is
    exactly why the lookup table was free: a wide residual stream gave
    the model plenty of room to memorize per-token gender regardless of
    how "small" the model looked by param count. A narrow tok_emb
    projected up into a wider residual stream still has to pass through
    that narrow bottleneck to get noun identity into the computation at
    all -- so THIS is the knob that should actually create a shared
    phase, not overall param count.
    """
    def __init__(self, vocab_size, n_embd, n_layer, n_head, max_len, n_embd_tok=None):
        super().__init__()
        n_embd_tok = n_embd_tok or n_embd
        self.tok_emb = nn.Embedding(vocab_size, n_embd_tok)
        self.tok_proj = (nn.Linear(n_embd_tok, n_embd, bias=False)
                          if n_embd_tok != n_embd else nn.Identity())
        self.pos_emb = nn.Parameter(torch.zeros(1, max_len, n_embd))
        self.blocks = nn.ModuleList([Block(n_embd, n_head, max_len) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
        self.max_len = max_len

    def forward(self, idx):
        B, T = idx.shape
        x = self.tok_proj(self.tok_emb(idx)) + self.pos_emb[:, :T, :]
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        return self.head(x)

    def num_params(self):
        return sum(p.numel() for p in self.parameters())

# Capacity grid: (n_layer, n_embd, n_head, n_embd_tok).
#
# n_embd_tok is the new, deliberately-narrow knob (see TinyGPT docstring)
# -- it's listed FIRST in spirit because it's the one doing the actual
# bottlenecking now. n_embd/n_head still vary too, both because the
# residual stream needs *some* room to do the grammar's other work
# (case, verb agreement, structure -- this is a genuinely demanding
# grammar, not a single-task toy) and because the theory's capacity
# variable was never meant to be "one specific tensor's width" -- but
# n_embd_tok is what determines whether the (lang, noun) -> gender table
# is cheap or expensive to store.
#
# The first run swept n_embd from 64 (178K params) up and got acc=1.000
# EVERYWHERE at every point -- confirming the lookup table is cheap for
# any n_embd_tok that large. This grid adds several points where
# n_embd_tok is small enough that memorizing 8 languages x 160 nouns'
# worth of gender assignments should compete directly against everything
# else the token embedding has to encode (noun's OTHER properties:
# nothing else, in this grammar -- but also against fitting 8 languages'
# worth of distinct token identities through the same narrow channel).
#
# Widen/narrow to taste -- the point is to bracket the predicted N*
# crossovers, not to match any particular real-model scale. Kept
# deliberately modest on n_embd/n_layer at the top end -- attention is
# O(T^2) in MAX_LEN and cost scales worse than linearly with depth x
# width jointly, so the last point or two can dominate total wall-clock
# time even though it's "one more entry in a list."
CAPACITY_GRID = [
    # (n_layer, n_embd, n_head, n_embd_tok)
    (1,  32,  1,   4),
    (1,  32,  1,   8),
    (2,  64,  2,  12),
    (2,  64,  2,  16),
    (2, 128,  4,  24),
    (2, 128,  4,  32),
    (4, 128,  4,  48),
    (4, 256,  8,  64),
    (6, 256,  8,  96),
]


# ══════════════════════════════════════════════════════════════════════
# CELL 8 — Training loop
# ══════════════════════════════════════════════════════════════════════

def train_model(n_layer, n_embd, n_head, langs, n_embd_tok=None, steps=3000, batch_size=256,
                 lr=6e-4, seed=0, log_every=500, use_dataloader=False):
    torch.manual_seed(seed)
    torch.backends.cudnn.benchmark = True  # fixed shapes every step (MAX_LEN constant) -- let cuDNN autotune once and reuse
    use_amp = device == "cuda"
    model = TinyGPT(VOCAB_SIZE, n_embd, n_layer, n_head, MAX_LEN, n_embd_tok=n_embd_tok).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    pad_id = STOI[PAD]

    if use_dataloader:
        loader = make_loader(langs, batch_size, seed_offset=seed, include_annots=False)
        it = iter(loader)
        def get_batch():
            ids, annots, batch_langs = next(it)
            return ids
    else:
        # Diagnostic path: bypass DataLoader/multiprocessing entirely --
        # sample directly in the main process, single-threaded. If a
        # stall that reproduces with the DataLoader disappears here, the
        # DataLoader/worker machinery is the cause, not the sampler or
        # the model/GPU side.
        rng = random.Random(seed)
        def get_batch():
            batch = [sample_packed_sequence(rng.choice(langs), rng)[0] for _ in range(batch_size)]
            return torch.tensor(batch, dtype=torch.long)

    model.train()
    t0 = time.time()
    for step in range(steps):
        ids = get_batch()
        ids = ids.to(device, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_amp):
            logits = model(ids[:, :-1])
            targets = ids[:, 1:].clone()
            targets[targets == pad_id] = -100
            loss = F.cross_entropy(logits.reshape(-1, VOCAB_SIZE), targets.reshape(-1), ignore_index=-100)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        if step % log_every == 0:
            print(f"  [N={model.num_params():,}] step {step:5d}  loss {loss.item():.4f}  "
                  f"({time.time()-t0:.0f}s elapsed)")
    return model


# ══════════════════════════════════════════════════════════════════════
# CELL 9 — Evaluation: restricted-vocab feature accuracy + Wilson CI
# ══════════════════════════════════════════════════════════════════════

def wilson_ci(successes, n, z=1.96):
    if n == 0:
        return (float('nan'), float('nan'))
    p = successes / n
    denom = 1 + z**2 / n
    centre = p + z**2 / (2 * n)
    adj = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))
    return ((centre - adj) / denom, (centre + adj) / denom)

@torch.no_grad()
def feature_accuracy_at_subcomp(model, lang, subcomp, n_eval=4000, batch_size=256, seed=999):
    """
    Restricted-vocab feature accuracy for a gender/class-bearing subcomp
    ('adj_agree', 'adj_agree_pre', 'v_agree', 'v_agree_obj'). Groups all
    valid tokens at that position by their target FEATURE (gender/class)
    and takes argmax over feature groups, not over the full vocabulary.

    Vectorized: matching (batch_row, position) pairs are gathered per
    eval batch and scored in ONE logsumexp call per feature group,
    instead of one torch call + .item() PER MATCHING POSITION. The
    per-position version forces a GPU-CPU sync at every single match
    (hundreds per batch) -- that's what was actually behind the GPU-util
    cliff right after training ended and eval started, not a real
    slowdown in the model or sampler.
    """
    model.eval()
    genders = genders_for(lang)
    if subcomp in ('adj_agree', 'adj_agree_pre'):
        groups = {g: [STOI[t] for t in ADJS[lang][g]] for g in genders}
    elif subcomp in ('v_agree', 'v_agree_obj'):
        spec = LANG_SPEC[lang]
        if spec['verb_agree'] != ('number', 'gender'):
            return None  # this subcomp doesn't carry a gender feature for this language
        groups = {}
        for g in genders:
            toks = []
            for n in ('SG', 'PL'):
                toks += VTR[lang][(g, n)] + VINTR[lang][(g, n)]
            groups[g] = [STOI[t] for t in toks]
    else:
        raise ValueError(subcomp)

    group_names = list(groups.keys())
    gender_to_idx = {g: i for i, g in enumerate(group_names)}
    # Index tensors built once, reused every batch -- no re-allocation per match.
    group_idx_tensors = [torch.tensor(groups[g], dtype=torch.long, device=device) for g in group_names]

    rng = random.Random(seed)
    score_chunks, label_chunks = [], []
    remaining = n_eval
    while remaining > 0:
        bs = min(batch_size, remaining)
        batch_ids, batch_ann = [], []
        for _ in range(bs):
            ids, ann, _ = sample_packed_sequence(lang, rng)
            batch_ids.append(ids)
            batch_ann.append(ann)
        ids_t = torch.tensor(batch_ids, dtype=torch.long, device=device)
        logits = model(ids_t[:, :-1])
        log_probs = F.log_softmax(logits, dim=-1)  # (B, T-1, V)

        # Cheap Python-side scan to find WHICH (row, pred_pos) pairs match --
        # this part is fine (dict lookups, no GPU calls). The GPU work is
        # deferred until every match in this batch is collected.
        rows, cols, labels = [], [], []
        for b, ann in enumerate(batch_ann):
            for pos, a in enumerate(ann):
                if a['subcomp'] != subcomp:
                    continue
                true_g = a.get('target_gender')
                if true_g is None:
                    continue
                pred_pos = pos - 1
                if pred_pos < 0:
                    continue
                rows.append(b)
                cols.append(pred_pos)
                labels.append(gender_to_idx[true_g])

        if rows:
            rows_t = torch.tensor(rows, device=device)
            cols_t = torch.tensor(cols, device=device)
            gathered = log_probs[rows_t, cols_t]  # (M, V) -- one gather, not M gathers
            # One logsumexp call PER GROUP (2 or 4 total), not per match.
            scores = torch.stack(
                [torch.logsumexp(gathered[:, idx], dim=1) for idx in group_idx_tensors], dim=1
            )  # (M, n_groups)
            score_chunks.append(scores.cpu())          # single sync per batch, not per match
            label_chunks.append(torch.tensor(labels))
        remaining -= bs

    if not score_chunks:
        return None
    all_scores = torch.cat(score_chunks, dim=0)
    all_labels = torch.cat(label_chunks, dim=0)
    preds = all_scores.argmax(dim=1)
    n_total = all_labels.numel()
    n_correct = (preds == all_labels).sum().item()
    acc = n_correct / n_total
    lo, hi = wilson_ci(n_correct, n_total)
    return dict(lang=lang, subcomp=subcomp, n=n_total, acc=acc, ci_lo=lo, ci_hi=hi)


def evaluate_capacity_point(model, langs, subcomps=('adj_agree', 'adj_agree_pre', 'v_agree', 'v_agree_obj')):
    results = []
    for lang in langs:
        for sc in subcomps:
            r = feature_accuracy_at_subcomp(model, lang, sc)
            if r is not None:
                results.append(r)
    return results


# ══════════════════════════════════════════════════════════════════════
# CELL 10 — Full sweep: train at every capacity point, every seed, evaluate
# ══════════════════════════════════════════════════════════════════════

def run_sweep(langs=LANGS, seeds=(0, 1), steps=3000,
              checkpoint_path='./v9_sweep_results.json',
              resume=True):
    """
    Returns a flat list of dicts: one row per (capacity, seed, lang, subcomp).
    Run this, then feed the output to extract_N_star below.
    Per the learnings file: always run >=2 seeds before trusting any
    single striking number -- this is enforced by the default `seeds`.

    Checkpointing: after EVERY (capacity, seed) point finishes, the
    accumulated rows are written to `checkpoint_path`. If `resume=True`
    and that file already exists, already-completed (capacity, seed)
    combinations are loaded and skipped rather than retrained -- so a
    killed or interrupted sweep can be restarted from the same command
    without losing finished runs. Completion is judged per (n_layer,
    n_embd, n_head, seed) tuple, not per individual eval row, since a
    given capacity/seed point either finished training+eval or it didn't.
    """
    rows = []
    done = set()
    if resume and os.path.exists(checkpoint_path):
        with open(checkpoint_path) as f:
            rows = json.load(f)
        done = {(r['n_layer'], r['n_embd'], r['n_head'], r.get('n_embd_tok', r['n_embd']), r['seed'])
                for r in rows}
        print(f"Resuming from checkpoint: {len(rows)} rows already recorded, "
              f"{len(done)} (capacity, seed) points already done.")

    for (n_layer, n_embd, n_head, n_embd_tok) in CAPACITY_GRID:
        for seed in seeds:
            key = (n_layer, n_embd, n_head, n_embd_tok, seed)
            if key in done:
                print(f"\n=== capacity (L={n_layer}, D={n_embd}, H={n_head}, Dtok={n_embd_tok})  seed={seed}  "
                      f"-- SKIPPED, already in checkpoint ===")
                continue
            print(f"\n=== capacity (L={n_layer}, D={n_embd}, H={n_head}, Dtok={n_embd_tok})  seed={seed} ===")
            model = train_model(n_layer, n_embd, n_head, langs, n_embd_tok=n_embd_tok, steps=steps, seed=seed)
            n_params = model.num_params()
            evals = evaluate_capacity_point(model, langs)
            for e in evals:
                e.update(n_params=n_params, n_layer=n_layer, n_embd=n_embd,
                          n_head=n_head, n_embd_tok=n_embd_tok, seed=seed)
                rows.append(e)
            print(f"  -- eval @ N={n_params:,} (Dtok={n_embd_tok}), seed={seed} --")
            for e in sorted(evals, key=lambda r: (r['subcomp'], r['lang'])):
                print(f"     {e['lang']:6s} {e['subcomp']:14s} acc={e['acc']:.3f} "
                      f"[{e['ci_lo']:.3f},{e['ci_hi']:.3f}]  n={e['n']}")
            del model
            torch.cuda.empty_cache()

            # Checkpoint immediately -- a run killed mid-sweep (as just
            # happened) should never cost you more than the ONE
            # (capacity, seed) point that was in flight.
            with open(checkpoint_path, 'w') as f:
                json.dump(rows, f, indent=2)
            print(f"  (checkpointed {len(rows)} total rows to {checkpoint_path})")

    return rows


# ══════════════════════════════════════════════════════════════════════
# CELL 11 — N* extraction and twin-pair comparison
# ══════════════════════════════════════════════════════════════════════

def crossover_capacity(rows, lang, subcomp, threshold=0.75, key='n_embd_tok'):
    """
    Empirical N* proxy: smallest evaluated capacity (by `key` -- default
    n_embd_tok, the narrow token-identity channel that actually gates the
    (lang, noun) -> gender lookup; use key='n_params' for the old,
    coarser total-parameter view) at which mean accuracy across seeds
    first exceeds `threshold`. Returns (capacity, accs_at_each_capacity)
    or (None, accs) if never crossed within the swept grid (consistent
    with the theory's own N*->infinity edge cases -- report this as a
    real result, not a failure).
    """
    by_capacity = defaultdict(list)
    for r in rows:
        if r['lang'] == lang and r['subcomp'] == subcomp:
            by_capacity[r.get(key, r['n_params'])].append(r['acc'])
    if not by_capacity:
        return None, {}
    means = {n: sum(v) / len(v) for n, v in sorted(by_capacity.items())}
    for n, m in means.items():
        if m >= threshold:
            return n, means
    return None, means

def summarize_twin_pairs(rows, subcomp='adj_agree', threshold=0.75, key='n_embd_tok'):
    print(f"\n--- N* summary at subcomp='{subcomp}', threshold={threshold}, capacity axis='{key}' ---")
    for base, twin, axis in TWIN_PAIRS:
        n_base, means_base = crossover_capacity(rows, base, subcomp, threshold, key=key)
        n_twin, means_twin = crossover_capacity(rows, twin, subcomp, threshold, key=key)
        print(f"\n[{axis}]")
        print(f"  {base:6s} N* = {n_base if n_base else '>max swept'}   "
              f"accs by {key}: " + ", ".join(f"{k}:{v:.2f}" for k, v in means_base.items()))
        print(f"  {twin:6s} N* = {n_twin if n_twin else '>max swept'}   "
              f"accs by {key}: " + ", ".join(f"{k}:{v:.2f}" for k, v in means_twin.items()))
        if n_base and n_twin:
            direction = "twin separates LATER (higher N*)" if n_twin > n_base else \
                        "twin separates EARLIER (lower N*)" if n_twin < n_base else \
                        "same N*"
            print(f"  -> {direction}")


# ══════════════════════════════════════════════════════════════════════
# CELL 12 — Entry point
# ══════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    # Sanity: print a handful of sampled sentences per language before
    # spending any compute, so a grammar bug is caught immediately.
    rng = random.Random(42)
    for L in LANGS:
        ids, ann, _ = sample_packed_sequence(L, rng)
        toks = [ITOS[i] for i in ids if ITOS[i] not in (PAD,)]
        print(f"\n[{L}] {' '.join(toks[:24])} ...")

    print("\nStarting capacity x seed sweep (checkpointed after every point --")
    print("safe to Ctrl+C and rerun this script to resume).")
    rows = run_sweep(seeds=(0, 1), steps=3000)

    print(f"\nDone. {len(rows)} total rows in ./v9_sweep_results.json")

    summarize_twin_pairs(rows, subcomp='adj_agree', threshold=0.75)
    summarize_twin_pairs(rows, subcomp='adj_agree_pre', threshold=0.75)
    summarize_twin_pairs(rows, subcomp='v_agree_obj', threshold=0.75)

Device: cuda  |  NUM_WORKERS: 16  (cpu_count=40)
Gender tables built and Hamming distances verified against spec.
Vocabulary: 893 tokens  |  Languages: ['A', 'B', 'C', 'Ctau', 'D', 'Dlo', 'Dpost', 'E']  |  Shared noun stems: 160
Running leakage audits (expect ~0.50 for all languages)...
[A] observed P(match)=0.506  expected-under-independence=0.504  (marginal={'F': 48326, 'M': 58036})  z=0.37  n=8000
[B] observed P(match)=0.506  expected-under-independence=0.507  (marginal={'F': 59039, 'M': 75075})  z=-0.12  n=8000
[C] observed P(match)=0.501  expected-under-independence=0.502  (marginal={'F': 49624, 'M': 56691})  z=-0.28  n=8000
[Ctau] observed P(match)=0.506  expected-under-independence=0.502  (marginal={'F': 49672, 'M': 56643})  z=0.73  n=8000
[D] observed P(match)=0.518  expected-under-independence=0.513  (marginal={'F': 45010, 'M': 61961})  z=0.91  n=8000
[Dlo] observed P(match)=0.506  expected-under-independence=0.504  (marginal={'M': 58153, 'F': 48818})  z=0.44  n=8000
[Dpost] o